# RQA of the ECG for Ventricular Fibrillation — SDDB

**Reproducibility notebook for the CBEB 2026 manuscript**
*"A Short-Scale Transition into Ventricular Fibrillation: Amplitude and Heart-Rate Variability Recurrence Contrast"*

Papani, Nunes, Soriano, Sales — UFPE / UFABC

---

This notebook contains **only** the analyses reported in the paper. Exploratory
work, discarded detectors, and unused databases were removed for clarity.

**Pipeline**
1. Setup & load results already saved to Drive (Part 0)
2. Data acquisition — SDDB (Part 1-2)
3. Core functions: filter, RQA (15 metrics), binarization (Part 3)
4. Filter selection test (Part 4)
5. Main RQA extraction, p05-p50 (Part 5)
6. Threshold/feature comparison (Part 6)
7. Predictive test: far vs near (Part 7)
8. Per-patient statistics — paired Cohen's d (Part 8)
9. HRV / Takens arm (Part 9)
10. Reviewer analyses: rhythm (R-5), same-7 contrast (R-4), gate coverage, bootstrap (Part 10)
11. Figure regeneration (Part 11)

> Data: SDDB is public at PhysioNet (ODC-BY). Raw signals are **not** included here.


## Part 0 — Setup and Loading Saved Results

In [ ]:
# =============================================================================
#  0.1 · AMBIENTE, CAMINHOS E ESTILO GLOBAL DAS FIGURAS
# =============================================================================
import os, sys, json, glob, time, shutil, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from scipy.spatial.distance import pdist, squareform
from scipy.signal import butter, filtfilt, iirnotch
from scipy.stats import mannwhitneyu, wilcoxon

# ---------- Monta o Drive (silencioso se já estiver montado) ----------
EM_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')
if EM_COLAB and not os.path.ismount('/content/drive'):
    from google.colab import drive
    drive.mount('/content/drive')

# ---------- Caminhos canônicos ----------
DRIVE_TCC = '/content/drive/MyDrive/TCC'

def _achar(base, *nomes):
    """Devolve a primeira subpasta existente (tolera maiúscula/minúscula)."""
    if not os.path.isdir(base):
        return os.path.join(base, nomes[0])
    existentes = {d.lower(): d for d in os.listdir(base)
                  if os.path.isdir(os.path.join(base, d))}
    for n in nomes:
        if n.lower() in existentes:
            return os.path.join(base, existentes[n.lower()])
    return os.path.join(base, nomes[0])

CSV_DIR    = _achar(DRIVE_TCC, 'csvs', 'csv', 'CSVs')
IMG_DIR    = _achar(DRIVE_TCC, 'Imagens', 'imagens', 'images')
MATRIZ_DIR = _achar(DRIVE_TCC, 'matrizes', 'Matrizes')
SDDB_DIR   = _achar(DRIVE_TCC, 'SDDB', 'sddb')

# ---------- Parâmetros globais do estudo ----------
FS         = 250            # Hz (SDDB)
JANELA_S   = 5              # s por janela de RQA
JANELA     = JANELA_S * FS  # 1250 amostras
MIN_ANTES  = 60             # min antes do vfon
L_MIN = V_MIN = 2           # comprimento mínimo de linha (padrão Marwan)
PERCENTIS  = [5, 10, 20, 30, 40, 50]
FEATURES   = ['RR','DET','L','L_max','DIV','L_entr','LAM','TT','V_max',
              'V_entr','W','W_max','W_entr','DET_RR','LAM_DET']

# =============================================================================
#  ESTILO GLOBAL DAS FIGURAS  —  legendas e rótulos legíveis, tudo em negrito
#  (afeta TODAS as figuras do notebook, inclusive as células antigas)
# =============================================================================
plt.rcParams.update({
    # resolução
    'figure.dpi'        : 110,
    'savefig.dpi'       : 200,
    'savefig.bbox'      : 'tight',
    'savefig.facecolor' : 'white',
    # tipografia — o principal ganho de legibilidade
    'font.size'         : 13,
    'axes.titlesize'    : 15,
    'axes.titleweight'  : 'bold',
    'axes.labelsize'    : 13,
    'axes.labelweight'  : 'bold',
    'xtick.labelsize'   : 11,
    'ytick.labelsize'   : 11,
    'figure.titlesize'  : 18,
    'figure.titleweight': 'bold',
    # legendas: fundo sólido para não sumir sobre a curva
    'legend.fontsize'   : 12,
    'legend.frameon'    : True,
    'legend.framealpha' : 0.93,
    'legend.edgecolor'  : '#333333',
    'legend.borderpad'  : 0.6,
    # traços e grade
    'lines.linewidth'   : 2.2,
    'axes.linewidth'    : 1.3,
    'axes.grid'         : True,
    'grid.alpha'        : 0.30,
    'grid.linestyle'    : '--',
    'axes.edgecolor'    : '#333333',
    'axes.labelcolor'   : '#111111',
})

def titulo_negrito(ax, texto, tam=15):
    """Atalho para padronizar títulos de subpainéis em negrito."""
    ax.set_title(texto, fontsize=tam, fontweight='bold', pad=8)
    return ax

print("=" * 78)
print("  AMBIENTE CONFIGURADO".center(78))
print("=" * 78)
for rot, cam in [('TCC', DRIVE_TCC), ('CSVs', CSV_DIR), ('Imagens', IMG_DIR),
                 ('Matrizes', MATRIZ_DIR), ('SDDB bruto', SDDB_DIR)]:
    print(f"  {'OK ' if os.path.isdir(cam) else '-- '} {rot:<12s} {cam}")
print(f"\n  Estilo de figuras aplicado: fonte {plt.rcParams['font.size']}pt, "
      f"títulos e eixos em NEGRITO, salvamento a {plt.rcParams['savefig.dpi']} dpi.")
print("=" * 78)

In [ ]:
# =============================================================================
#  0.2 · CARREGADOR UNIVERSAL — puxa TUDO que já está salvo no Drive
#        (CSVs -> DataFrames | Imagens -> índice | Matrizes -> índice | JSON)
#        Nada é reprocessado. Só leitura.
# =============================================================================

CSV        = {}   # nome_curto -> DataFrame
MANIFESTOS = {}   # percentil (int) -> DataFrame do manifesto
IMG        = {}   # nome_curto -> caminho do .png
MATRIZES   = {}   # record (str) -> {'pasta':..., 'arquivos':[...], 'n':int}
VFON       = {}   # record (str) -> vfon em segundos
PACIENTES_VFON = []

# ------------------------------------------------------------------ CSVs -----
def carregar_csvs(pasta=CSV_DIR):
    """Lê todo .csv da pasta. Manifestos vão também para MANIFESTOS[percentil]."""
    achados = sorted(glob.glob(os.path.join(pasta, '*.csv')))
    if not achados and os.path.isdir(DRIVE_TCC):
        # fallback: CSVs soltos na raiz de TCC (layout antigo)
        achados = sorted(glob.glob(os.path.join(DRIVE_TCC, '*.csv')))
    for cam in achados:
        nome = os.path.splitext(os.path.basename(cam))[0]
        try:
            df = pd.read_csv(cam)
        except Exception as e:
            print(f"  !! falha ao ler {nome}.csv: {type(e).__name__}: {e}")
            continue
        CSV[nome] = df
        if nome.startswith('manifesto_p'):
            try:
                MANIFESTOS[int(nome.split('_p')[1])] = df
            except ValueError:
                pass
    return CSV

# --------------------------------------------------------------- Imagens -----
def carregar_imagens(pasta=IMG_DIR):
    for cam in sorted(glob.glob(os.path.join(pasta, '*.png'))
                      + glob.glob(os.path.join(pasta, '*.jpg'))):
        IMG[os.path.splitext(os.path.basename(cam))[0]] = cam
    return IMG

# -------------------------------------------------------------- Matrizes -----
def carregar_indice_matrizes(pasta=MATRIZ_DIR):
    """Indexa (sem carregar em RAM) as matrizes .npz/.npy de cada paciente."""
    if not os.path.isdir(pasta):
        return MATRIZES
    for rec in sorted(os.listdir(pasta)):
        sub = os.path.join(pasta, rec)
        if not os.path.isdir(sub):
            continue
        arqs = sorted([a for a in os.listdir(sub) if a.endswith(('.npz', '.npy'))])
        if arqs:
            MATRIZES[rec] = {'pasta': sub, 'arquivos': arqs, 'n': len(arqs)}
    return MATRIZES

def abrir_matriz(record, indice=0):
    """Carrega UMA matriz sob demanda (as matrizes NÃO ficam todas em RAM)."""
    if record not in MATRIZES:
        raise KeyError(f"Paciente {record} não indexado. Disponíveis: {list(MATRIZES)}")
    cam = os.path.join(MATRIZES[record]['pasta'], MATRIZES[record]['arquivos'][indice])
    dados = np.load(cam)
    if cam.endswith('.npz'):
        return {k: dados[k] for k in dados.files}
    return dados

# ------------------------------------------------------- pacientes_com_vfon --
def carregar_vfon():
    """Procura pacientes_com_vfon.json nos locais plausíveis e lê os vfon dos headers."""
    global PACIENTES_VFON
    candidatos = [os.path.join(p, 'pacientes_com_vfon.json')
                  for p in (MATRIZ_DIR, SDDB_DIR, DRIVE_TCC, CSV_DIR, '.')]
    for cam in candidatos:
        if os.path.exists(cam):
            with open(cam) as f:
                PACIENTES_VFON = json.load(f)
            print(f"  pacientes_com_vfon.json  <- {cam}  ({len(PACIENTES_VFON)} registros)")
            break
    else:
        print("  pacientes_com_vfon.json  não encontrado (será gerado na Parte 1).")

    # vfon em segundos, lido dos headers disponíveis
    for pasta in (MATRIZ_DIR, SDDB_DIR, '.'):
        for hea in sorted(glob.glob(os.path.join(pasta, '**', '*.hea'), recursive=True)):
            rec = os.path.splitext(os.path.basename(hea))[0]
            if rec in VFON:
                continue
            try:
                with open(hea, errors='ignore') as f:
                    for linha in f:
                        if 'vfon' in linha.lower():
                            hh, mm, ss = map(int, linha.split(':', 1)[1].strip().split(':'))
                            VFON[rec] = hh * 3600 + mm * 60 + ss
                            break
            except Exception:
                pass
    return PACIENTES_VFON, VFON

# ------------------------------------------------------------- EXECUÇÃO ------
print("Carregando o que já está salvo no Drive...\n")
carregar_csvs()
carregar_imagens()
carregar_indice_matrizes()
carregar_vfon()
print("\nCarregamento concluído. Rode a próxima célula para ver o inventário.")

In [ ]:
# =============================================================================
#  0.3 · INVENTÁRIO DO QUE FOI CARREGADO
# =============================================================================
from IPython.display import display, Markdown

def _bloco(titulo):
    print("\n" + "=" * 78)
    print(f"  {titulo}".ljust(78))
    print("=" * 78)

# ---------------------------------------------------------------- CSVs -------
_bloco(f"CSVs CARREGADOS  ({len(CSV)})")
if CSV:
    inv = pd.DataFrame([{'arquivo': f'{k}.csv', 'linhas': len(v), 'colunas': v.shape[1],
                         'primeiras colunas': ', '.join(map(str, v.columns[:5]))}
                        for k, v in sorted(CSV.items())])
    display(inv.style.set_properties(**{'text-align': 'left'})
                     .set_table_styles([{'selector': 'th',
                                         'props': [('font-weight', 'bold'),
                                                   ('text-align', 'left')]}])
                     .hide(axis='index'))
    print(f"  Acesso:  CSV['teste_predicao']   |   MANIFESTOS[40]")
    print(f"  Manifestos disponíveis (percentis): {sorted(MANIFESTOS)}")
else:
    print("  Nenhum CSV encontrado. Confira se a pasta é TCC/csvs.")

# ------------------------------------------------------------ Imagens -------
_bloco(f"IMAGENS INDEXADAS  ({len(IMG)})")
for nome in sorted(IMG):
    print(f"  - {nome}.png")
print("\n  Acesso:  IMG['trajetorias_15_features_p40']")

# ----------------------------------------------------------- Matrizes -------
_bloco(f"MATRIZES DE DISTÂNCIA INDEXADAS  ({len(MATRIZES)} pacientes)")
if MATRIZES:
    tot = sum(v['n'] for v in MATRIZES.values())
    inv_m = pd.DataFrame([{'paciente': r, 'arquivos': v['n'],
                           'vfon (s)': VFON.get(r, '—'),
                           'exemplo': v['arquivos'][0]}
                          for r, v in sorted(MATRIZES.items(), key=lambda x: int(x[0]))])
    display(inv_m.style.set_table_styles(
        [{'selector': 'th', 'props': [('font-weight', 'bold')]}]).hide(axis='index'))
    print(f"  Total de arquivos de matriz: {tot}")
    print(f"  Acesso sob demanda:  m = abrir_matriz('30', 0)   # NÃO carrega tudo em RAM")
else:
    print("  Nenhuma matriz indexada. Confira a pasta TCC/matrizes.")

# ------------------------------------------------------------- Resumo -------
_bloco("RESUMO")
print(f"  Pacientes com vfon anotado : {len(PACIENTES_VFON)}  {PACIENTES_VFON}")
print(f"  Headers com vfon lido      : {len(VFON)}")
print(f"  Percentis de binarização   : {PERCENTIS}")
print(f"  15 features do RQA         : {', '.join(FEATURES)}")
print("=" * 78)

### 0.1 Figure gallery (optional)

In [ ]:
# =============================================================================
#  0.4 · GALERIA DAS FIGURAS JÁ SALVAS  (não reprocessa nada)
# =============================================================================
from IPython.display import Image as _Img, display, Markdown

# Escolha o que exibir: 'trajetorias', 'boxplots' ou 'todas'
MOSTRAR = 'todas'
LARGURA = 1100          # px

def mostrar_figuras(filtro='todas', largura=LARGURA):
    nomes = sorted(IMG)
    if filtro != 'todas':
        nomes = [n for n in nomes if filtro in n]
    if not nomes:
        print("Nenhuma figura para exibir.")
        return
    for n in nomes:
        display(Markdown(f"#### `{n}.png`"))
        display(_Img(filename=IMG[n], width=largura))

mostrar_figuras(MOSTRAR)

## Part 1 — Libraries and Data Acquisition

In [ ]:
# Instala dependências no Google Colab
!pip install wfdb --quiet

import os, time, shutil, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import wfdb
from scipy.spatial.distance import pdist, squareform
from scipy.signal import butter, filtfilt, iirnotch
from scipy.stats import mannwhitneyu, wilcoxon
from itertools import groupby

# ---- MONTA O DRIVE (obrigatório antes de criar qualquer pasta) ----
from google.colab import drive
if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')

# trava de segurança: só continua se o Drive REAL estiver montado
if not os.path.ismount('/content/drive'):
    raise RuntimeError(
        "❌ Google Drive NÃO está montado. Sem isso, as pastas seriam criadas no disco "
        "temporário (fantasma) e os dados se perderiam ao reiniciar.\n"
        "   Monte pelo ícone de pasta na barra lateral > 'Mount Drive', ou rode drive.mount()."
    )
print("✅ Drive montado de verdade:", os.path.ismount('/content/drive'))

# ---- Parâmetros globais ----
FS          = 250
JANELA_S    = 5
JANELA      = JANELA_S * FS
MIN_ANTES   = 60
L_MIN = V_MIN = 2

# ---- Pastas de saída (agora seguras, pois o Drive está montado) ----
DRIVE_TCC   = '/content/drive/MyDrive/TCC'
DRIVE_MATRIZ= os.path.join(DRIVE_TCC, 'matrizes')
os.makedirs(DRIVE_TCC, exist_ok=True)

# confirma que a pasta TCC do Drive REAL tem conteúdo
conteudo = os.listdir(DRIVE_TCC)
print(f"📁 {DRIVE_TCC} contém {len(conteudo)} itens:", conteudo[:10])
print("Ambiente configurado.")

In [ ]:
import os, json, shutil
import wfdb

# --- 1. MONTA O DRIVE (sem quebrar se já estiver montado) ---
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')
else:
    print("Drive já acessível ✅")

DRIVE_DIR = '/content/drive/MyDrive/TCC/SDDB'   # cache permanente (sobrevive ao reset do Colab)
LOCAL_DIR = '.'                                  # leitura rápida na sessão atual
os.makedirs(DRIVE_DIR, exist_ok=True)

database_name = 'sddb'
CACHE_VFON = f'{DRIVE_DIR}/pacientes_com_vfon.json'

print(f"\n--- OBTENDO A LISTA DE REGISTROS DA BASE {database_name.upper()} ---")
lista_pacientes = wfdb.get_record_list(database_name)
print(f"Total de registros encontrados: {len(lista_pacientes)} pacientes.\n")

def tem_arquivos(pasta, rec):
    """Verifica se o .hea E o .dat do registro já existem na pasta."""
    return os.path.exists(f'{pasta}/{rec}.hea') and os.path.exists(f'{pasta}/{rec}.dat')

pacientes_com_vfon = []

# --- 2. LOOP: RECUPERA DO DRIVE OU BAIXA DO PHYSIONET ---
for record_id in lista_pacientes:
    print(f"--- PROCESSANDO PACIENTE {record_id} ---")
    try:
        if tem_arquivos(DRIVE_DIR, record_id):
            for ext in ('hea', 'dat'):
                shutil.copy(f'{DRIVE_DIR}/{record_id}.{ext}', LOCAL_DIR)
            print(f"  ♻️  Recuperado do Drive (sem re-download)")
        else:
            wfdb.dl_database(database_name, dl_dir=LOCAL_DIR, records=[record_id], overwrite=True)
            for ext in ('hea', 'dat'):
                shutil.copy(f'{LOCAL_DIR}/{record_id}.{ext}', DRIVE_DIR)
            print(f"  ⬇️  Baixado do PhysioNet e salvo no Drive")

        header = wfdb.rdheader(record_id)
        print(f"  Frequência: {header.fs} Hz | Canais: {header.sig_name} | Pontos: {header.sig_len}")
        for comentario in header.comments:
            if 'vfon' in comentario.lower():
                print(f"  -> ⚠️ FIBRILAÇÃO ENCONTRADA: {comentario}")
                pacientes_com_vfon.append(record_id)
                break
    except Exception as e:
        print(f"  -> ❌ Erro ao processar o registro {record_id}: {e}")
    print("-" * 50)

# --- 3. SALVA A LISTA DE ELEGÍVEIS NO DRIVE ---
with open(CACHE_VFON, 'w') as f:
    json.dump(pacientes_com_vfon, f)

print("\n" + "=" * 50)
print("✅ PROCESSAMENTO CONCLUÍDO!")
print(f"Dos {len(lista_pacientes)} pacientes, {len(pacientes_com_vfon)} possuem anotação de Fibrilação (vfon).")
print(f"Lista de pacientes elegíveis: {pacientes_com_vfon}")
print(f"💾 Lista salva em: {CACHE_VFON}")
print("=" * 50)

## Part 2 — SDDB Database

### 2.1. SDDB — Sudden Cardiac Death Holter Database

* **Conteúdo:** Gravações de ECG (Holter) de longo prazo de pacientes que sofreram morte súbita cardíaca / fibrilação ventricular durante o monitoramento.
* **Volume:** 23 gravações completas.
* **Ficha técnica (registro 30):**
  * Taxa de amostragem: 250 Hz
  * Canais: 2 (['ECG', 'ECG'])
  * Total de pontos: 22.099.250
  * Duração: 24 h 33 min 17 s
  * Anotação clínica: segundo exato do início da FV (ex.: `vfon: 07:54:33`)

> **Nota metodológica sobre o `vfon`:** verificamos que a anotação representa **tempo decorrido desde o início da gravação** (não hora do relógio). Prova: no registro 38, a interpretação como hora do relógio colocaria o evento fora da duração gravada. Portanto `amostra = segundos_vfon × fs` está correto.

In [ ]:
# Lista os registros da SDDB e identifica quais têm anotação de FV (vfon)
lista_sddb = wfdb.get_record_list('sddb')
print(f"SDDB: {len(lista_sddb)} registros.")

## Part 3 — Core Functions: Filter, RQA, Binarization

In [ ]:
def limpar_nan(sinal):
    """Interpola linearmente valores NaN/inf. Essencial antes do filtfilt."""
    sinal = np.asarray(sinal, dtype=float)
    ruim = ~np.isfinite(sinal)
    if ruim.any():
        idx = np.arange(len(sinal)); ok = ~ruim
        if ok.sum() < 2:
            return np.zeros_like(sinal)
        sinal = sinal.copy()
        sinal[ruim] = np.interp(idx[ruim], idx[ok], sinal[ok])
    return sinal

def filtrar_ecg(sinal, fs=FS, lowcut=0.5, highcut=40.0, ordem=3):
    """
    Butterworth passa-banda, fase zero (filtfilt), robusto a NaN.
    Cortes padrão 0.5-40 Hz: escolha JUSTIFICADA pelo teste da seção 4.
    """
    sinal = limpar_nan(sinal)
    nyq = 0.5 * fs
    b, a = butter(ordem, [lowcut/nyq, highcut/nyq], btype='band')
    return filtfilt(b, a, sinal)

# validação rápida
_t = np.array([1.0, 2.0, np.nan, 4.0, 5.0] * 250)
print("Filtro trata NaN?", not np.isnan(filtrar_ecg(_t)).any())

In [ ]:
def _runs(arr, valor):
    """Comprimentos de sequências consecutivas iguais a `valor`."""
    mask = (np.asarray(arr) == valor).astype(np.int8)
    if mask.size == 0:
        return np.empty(0, dtype=int)
    d = np.diff(np.concatenate(([0], mask, [0])))
    return np.where(d == -1)[0] - np.where(d == 1)[0]

def _entropia(lengths):
    if len(lengths) == 0:
        return 0.0
    _, counts = np.unique(lengths, return_counts=True)
    p = counts / counts.sum()
    return float(-np.sum(p * np.log(p)))

def calcular_rqa_completo(rp, l_min=2, v_min=2):
    """15 métricas do RQA a partir da matriz de recorrência binária rp (NxN)."""
    N = rp.shape[0]; rp = rp.astype(np.int8)
    RR = rp.sum() / (N * N)

    # Diagonais (exclui LOI)
    dl = [_runs(np.diagonal(rp, k), 1) for k in range(1, N)]
    dl = np.concatenate(dl) if dl else np.empty(0, dtype=int)
    tot_d = dl.sum(); dge = dl[dl >= l_min]
    DET   = (dge.sum()/tot_d*100) if tot_d > 0 else 0.0
    L     = dge.mean() if len(dge) else 0.0
    L_max = int(dl.max()) if len(dl) else 0
    DIV   = (1.0/L_max) if L_max > 0 else 0.0
    L_entr= _entropia(dge)

    # Verticais e verticais brancas
    vl, wl = [], []
    for j in range(N):
        col = rp[:, j]; vl.append(_runs(col, 1)); wl.append(_runs(col, 0))
    vl = np.concatenate(vl); wl = np.concatenate(wl)
    tot_v = vl.sum(); vge = vl[vl >= v_min]
    LAM   = (vge.sum()/tot_v*100) if tot_v > 0 else 0.0
    TT    = vge.mean() if len(vge) else 0.0
    V_max = int(vl.max()) if len(vl) else 0
    V_entr= _entropia(vge)
    wge = wl[wl >= v_min]
    W     = wge.mean() if len(wge) else 0.0
    W_max = int(wl.max()) if len(wl) else 0
    W_entr= _entropia(wge)

    DET_RR  = (DET/100)/RR if RR > 0 else 0.0
    LAM_DET = (LAM/100)/(DET/100) if DET > 0 else 0.0
    return {'RR':RR,'DET':DET,'L':L,'L_max':L_max,'DIV':DIV,'L_entr':L_entr,
            'LAM':LAM,'TT':TT,'V_max':V_max,'V_entr':V_entr,
            'W':W,'W_max':W_max,'W_entr':W_entr,'DET_RR':DET_RR,'LAM_DET':LAM_DET}

# teste de sanidade: seno puro -> DET alto
_t = np.sin(np.linspace(0, 10*np.pi, 1250))
_m = squareform(pdist(_t.reshape(-1,1)))
_rp = (_m <= np.percentile(_m[_m>0], 10)).astype(np.int8)
print("Seno puro -> DET =", round(calcular_rqa_completo(_rp)['DET'],1), "(esperado: alto)")

In [ ]:
def binarizar(mat, percentil):
    """Retorna a matriz de recorrência binária no percentil dado (robusta a escala)."""
    dpos = mat[mat > 0]
    if dpos.size == 0:
        return None
    lim = np.percentile(dpos, percentil)
    return (mat <= lim).astype(np.int8)

print("Função de binarização pronta.")

## Part 4 — Filter Selection Test

In [ ]:
def cohen_d(a, b):
    """Tamanho de efeito. |d|: 0.2 pequeno, 0.5 médio, 0.8 grande."""
    a, b = np.asarray(a), np.asarray(b)
    if len(a) < 2 or len(b) < 2: return np.nan
    sp = np.sqrt(((len(a)-1)*a.std(ddof=1)**2 + (len(b)-1)*b.std(ddof=1)**2)/(len(a)+len(b)-2))
    return (a.mean()-b.mean())/sp if sp > 0 else np.nan

# Variantes de filtro
def _hp(x, fc):  # passa-alta
    nyq=0.5*FS; b,a=butter(3, fc/nyq, btype='high'); return filtfilt(b,a,limpar_nan(x))
def _bp(x, lo, hi):  # passa-banda
    nyq=0.5*FS; b,a=butter(3,[lo/nyq,hi/nyq],btype='band'); return filtfilt(b,a,limpar_nan(x))
def _bp_notch(x):
    y=_bp(x,1,50); nyq=0.5*FS; bn,an=iirnotch(60/nyq,Q=30); return filtfilt(bn,an,y)

FILTROS = {
    'sem_filtro':        lambda x: limpar_nan(x),
    'butter_1_50':       lambda x: _bp(x,1,50),
    'butter_0.5_40':     lambda x: _bp(x,0.5,40),
    'butter_1_50+notch': _bp_notch,
    'so_baseline_0.5hp': lambda x: _hp(x,0.5),
}

FEATURES = ['RR','DET','L','L_max','DIV','L_entr','LAM','TT','V_max',
            'V_entr','W','W_max','W_entr','DET_RR','LAM_DET']

def rqa_de(janela, percentil=10):
    if janela.std()==0: return None
    mat = squareform(pdist(janela.reshape(-1,1), metric='euclidean'))
    rp = binarizar(mat, percentil)
    return None if rp is None else calcular_rqa_completo(rp, L_MIN, V_MIN)

print("Funções do teste de filtro prontas. Rode a próxima célula para executar o teste.")

In [ ]:
N_TESTE = 23
MIN_ANTES = 60          # 60 min ANTES do vfon (sem período depois)
dados_filtro = {nome: [] for nome in FILTROS}
processados = 0

for rec in [str(r) for r in wfdb.get_record_list('sddb')[:N_TESTE]]:
    try:
        header = wfdb.rdheader(rec)
    except FileNotFoundError:
        print(f"⚠️ {rec} não local. Pulando."); continue
    ev = next((c for c in header.comments if 'vfon' in c.lower()), None)
    if not ev:
        print(f"⏭️ {rec} sem vfon."); continue
    h, m, s = map(int, ev.split(': ')[1].strip().split(':'))
    vfon = ((h*3600)+(m*60)+s)*FS
    ini = max(0, vfon - MIN_ANTES*60*FS); fim = vfon
    bruto = wfdb.rdsamp(rec, sampfrom=ini, sampto=fim)[0][:, 0]
    filt = {nome: fn(bruto) for nome, fn in FILTROS.items()}
    n = len(bruto)//JANELA
    print(f"  {rec}: {n} janelas")
    processados += 1
    for w in range(n):
        i = w*JANELA; t_rel=(ini+i-vfon)/FS
        faixa = 'longe' if t_rel < -30*60 else 'perto'   # longe/perto do colapso
        for nome, sig in filt.items():
            f = rqa_de(sig[i:i+JANELA])
            if f: f['faixa']=faixa; dados_filtro[nome].append(f)

if processados == 0:
    raise RuntimeError("❌ Nenhum paciente processado — rode a célula 1.1 (cache/download) antes.")

linhas = []
for nome in FILTROS:
    df = pd.DataFrame(dados_filtro[nome])
    if df.empty: continue
    ds = [(feat, abs(cohen_d(df[df.faixa=='longe'][feat].dropna(),
                             df[df.faixa=='perto'][feat].dropna()))) for feat in FEATURES]
    ds = [(f,d) for f,d in ds if not np.isnan(d)]
    if not ds: continue
    top = max(ds, key=lambda x: x[1])
    linhas.append({'filtro':nome,'melhor_feature':top[0],
                   'cohen_d_max':round(top[1],3),
                   'cohen_d_medio':round(np.mean([d for _,d in ds]),3)})

df_filtros = pd.DataFrame(linhas).sort_values('cohen_d_max', ascending=False)
df_filtros.to_csv(os.path.join(DRIVE_TCC,'comparacao_filtros.csv'), index=False)
print("\n=== COMPARAÇÃO DE FILTROS (longe vs perto do colapso) ===")
print(df_filtros.to_string(index=False))
print(f"\n🏆 Melhor: {df_filtros.iloc[0].filtro} (d={df_filtros.iloc[0].cohen_d_max})")
print("💾 Salvo: comparacao_filtros.csv")

In [ ]:
def filtrar_ecg(sinal, fs=FS, lowcut=0.5, highcut=40, ordem=3, notch60=True):
    """
    Butterworth passa-banda 1-50 Hz (melhor no teste de filtros) + notch 60 Hz opcional.
    Fase zero (filtfilt), robusto a NaN.
    """
    sinal = limpar_nan(sinal)
    nyq = 0.5 * fs
    b, a = butter(ordem, [lowcut/nyq, highcut/nyq], btype='band')
    y = filtfilt(b, a, sinal)
    if notch60:                      # remove ruído de rede elétrica (60 Hz, EUA)
        bn, an = iirnotch(60/nyq, Q=30)
        y = filtfilt(bn, an, y)
    return y

# valida
_t = np.array([1.0, 2.0, np.nan, 4.0, 5.0] * 250)
print("Filtro atualizado (1-50 Hz + notch 60). Trata NaN?", not np.isnan(filtrar_ecg(_t)).any())

## Part 5 — Main RQA Extraction (p05-p50)

In [ ]:
import os, time
import numpy as np
import pandas as pd
import wfdb
from scipy.spatial.distance import pdist, squareform
from tqdm.auto import tqdm   # barra de progresso (auto = bonita no Colab)

# ---- Config do processamento ----
N_PACIENTES = 23
PERCENTIS   = [5, 10, 20, 30, 40, 50]
SALVAR_MATRIZ = True
COMPACTAR   = True
LIMPAR_ANTES  = True
MIN_ANTES   = 60           # 60 min ANTES do vfon
MIN_POS     = 5            # 5 min DEPOIS do vfon (FV instalada, p/ comparar)

if LIMPAR_ANTES:
    for p in PERCENTIS:
        f = os.path.join(DRIVE_TCC, f'manifesto_p{p:02d}.csv')
        if os.path.exists(f): os.remove(f)
if SALVAR_MATRIZ:
    os.makedirs(DRIVE_MATRIZ, exist_ok=True)

registros = {p: [] for p in PERCENTIS}
tempos = []

lista = [str(r) for r in wfdb.get_record_list('sddb')[:N_PACIENTES]]

# --- BARRA DE PROGRESSO por paciente ---
barra = tqdm(lista, desc="Processando", unit="paciente")
for rec in barra:
    barra.set_description(f"Paciente {rec}")   # mostra o paciente atual na barra
    t0 = time.time()
    try: header = wfdb.rdheader(rec)
    except FileNotFoundError:
        tqdm.write(f"⚠️ {rec} não local. Pulando."); continue
    ev = next((c for c in header.comments if 'vfon' in c.lower()), None)
    if not ev:
        tqdm.write(f"⏭️ {rec} sem vfon."); continue
    h,m,s = map(int, ev.split(': ')[1].strip().split(':'))
    vfon = ((h*3600)+(m*60)+s)*FS

    ini = max(0, vfon - MIN_ANTES*60*FS)
    fim = min(header.sig_len, vfon + MIN_POS*60*FS)
    if vfon - MIN_ANTES*60*FS < 0:
        tqdm.write(f"  ⚠️ {rec}: <{MIN_ANTES}min antes (recorte encurtado no início).")
    if vfon + MIN_POS*60*FS > header.sig_len:
        tqdm.write(f"  ⚠️ {rec}: <{MIN_POS}min depois (recorte encurtado no fim).")

    ecg = filtrar_ecg(wfdb.rdsamp(rec, sampfrom=ini, sampto=fim)[0][:,0], fs=FS)
    n = len(ecg)//JANELA
    pasta = os.path.join(DRIVE_MATRIZ, rec)
    if SALVAR_MATRIZ: os.makedirs(pasta, exist_ok=True)

    lote_matrizes = {}
    salvas=puladas=zeradas=0
    # barra interna opcional das janelas (leave=False some ao terminar)
    for w in tqdm(range(n), desc=f"  {rec} janelas", leave=False, unit="jan"):
        i = w*JANELA; ini_abs = ini+i
        jan = ecg[i:i+JANELA]
        if jan.std()==0: puladas+=1; continue
        mat = squareform(pdist(jan.reshape(-1,1), metric='euclidean')).astype(np.float32)
        if mat[mat>0].size==0: puladas+=1; continue

        t_rel = (ini_abs - vfon)/FS
        t_min = t_rel/60

        if t_rel < 0:
            faixa = 'longe' if t_rel < -30*60 else 'perto'
        else:
            faixa = 'pos'

        marca = 'POS' if t_rel >= 0 else 'ANTES'
        chave = f"{rec}_w{w:04d}_{marca}{abs(t_min):04.1f}min"

        if SALVAR_MATRIZ:
            if COMPACTAR:
                lote_matrizes[chave] = mat
            else:
                np.save(os.path.join(pasta, chave+'.npy'), mat)
            salvas+=1

        for p in PERCENTIS:
            rp = binarizar(mat, p)
            feats = calcular_rqa_completo(rp, L_MIN, V_MIN)
            if feats['RR']==0: zeradas+=1
            feats.update({'record':int(rec),'window':w,'amostra_ini':ini_abs,
                          'tempo_rel_vfon_s':t_rel,'tempo_rel_vfon_min':round(t_min,2),
                          'faixa':faixa,'percentil':p,
                          'arquivo':(chave+'.npy') if SALVAR_MATRIZ else ''})
            registros[p].append(feats)

    if SALVAR_MATRIZ and COMPACTAR and lote_matrizes:
        # o savez no Drive é lento -> mostra que está salvando
        barra.set_postfix_str("salvando .npz...")
        np.savez_compressed(os.path.join(pasta, f'{rec}_matrizes.npz'), **lote_matrizes)

    dt=time.time()-t0; tempos.append(dt)
    n_pos = sum(1 for k in (lote_matrizes or {}) if 'POS' in k)
    # log detalhado via tqdm.write (não quebra a barra)
    tqdm.write(f"  ✅ {rec}: janelas={n} (POS={n_pos}) salvas={salvas} "
               f"puladas={puladas} zerados={zeradas} ⏱️{dt:.1f}s")
    # atualiza o postfix da barra com um resumo vivo
    barra.set_postfix_str(f"últ: {rec} ({dt:.0f}s)")

    for p in PERCENTIS:
        pd.DataFrame(registros[p]).to_csv(os.path.join(DRIVE_TCC,f'manifesto_p{p:02d}.csv'), index=False)

print("\n✅ Concluído. Matrizes em:", DRIVE_MATRIZ)
if tempos:
    seg = np.mean(tempos)
    print(f"Tempo médio/paciente: {seg:.1f}s → total 23: {seg*23/60:.1f} min")
    jpp = (MIN_ANTES+MIN_POS)*60//JANELA_S
    mb = jpp * 6.25
    print(f"Espaço (sem compressão): ~{mb/1000:.1f} GB/paciente → ~{mb*23/1000:.0f} GB total")

In [ ]:
import os, numpy as np

# 1. SALVAR_MATRIZ está True mesmo?
print("SALVAR_MATRIZ =", SALVAR_MATRIZ)

# 2. O Drive está montado e a pasta existe/tem permissão de escrita?
print("DRIVE_BASE =", DRIVE_BASE)
print("Pasta existe?", os.path.exists(DRIVE_BASE))
print("Drive montado?", os.path.exists('/content/drive/MyDrive'))

# 3. Teste de escrita real: consegue salvar um .npy de teste?
try:
    os.makedirs(os.path.join(DRIVE_BASE, '30'), exist_ok=True)
    teste = np.zeros((10, 10), dtype=np.float32)
    caminho_teste = os.path.join(DRIVE_BASE, '30', 'TESTE.npy')
    np.save(caminho_teste, teste)
    print("✅ Escrita OK:", caminho_teste, "existe?", os.path.exists(caminho_teste))
    os.remove(caminho_teste)
except Exception as e:
    print("❌ FALHA ao escrever:", type(e).__name__, e)

## Part 6 — Threshold and Feature Comparison

In [ ]:
def comparar_manifestos(diretorio=DRIVE_TCC, percentis=PERCENTIS):
    resultados=[]; saude=[]
    for p in percentis:
        caminho=os.path.join(diretorio,f'manifesto_p{p:02d}.csv')
        if not os.path.exists(caminho): continue
        df=pd.read_csv(caminho)
        saude.append({'percentil':p,'linhas':len(df),
                      'RR_zerado_%':round((df.RR==0).mean()*100,1),
                      'n_longe':(df.faixa=='longe').sum(),'n_perto':(df.faixa=='perto').sum()})
        for feat in FEATURES:
            a=df[df.faixa=='longe'][feat].dropna(); b=df[df.faixa=='perto'][feat].dropna()
            if len(a)<5 or len(b)<5: continue
            d=abs(cohen_d(a,b))
            try: _,pv=mannwhitneyu(a,b)
            except ValueError: pv=np.nan
            resultados.append({'percentil':p,'feature':feat,'cohen_d':round(d,3),
                               'p_valor':pv,'media_longe':round(a.mean(),3),
                               'media_perto':round(b.mean(),3)})
    return pd.DataFrame(resultados), pd.DataFrame(saude)

res, saude = comparar_manifestos()
print("=== SANIDADE (RR_zerado deve ser ~0) ===")
print(saude.to_string(index=False))
print("\n=== TOP 15 (feature × limiar) — separação longe vs perto ===")
print(res.sort_values('cohen_d', ascending=False).head(15).to_string(index=False))
res.to_csv(os.path.join(DRIVE_TCC,'comparacao_features_limiares.csv'), index=False)
print("\n💾 Salvo: comparacao_features_limiares.csv")
print("\n=== RANKING DE LIMIARES ===")
print(res.groupby('percentil')['cohen_d'].agg(['mean','max']).round(3).sort_values('mean',ascending=False).to_string())
print("\n=== RANKING DE FEATURES ===")
print(res.groupby('feature')['cohen_d'].mean().round(3).sort_values(ascending=False).to_string())

## Part 7 — Predictive Test: far vs near

In [ ]:
df = pd.read_csv(os.path.join(DRIVE_TCC,'manifesto_p05.csv'))

# --- Evolução temporal de uma feature (bins de 5 min) ---
print("=== Evolução temporal no período pré-colapso (percentil 40) ===")
faixas=[(-60,-50),(-50,-40),(-40,-30),(-30,-20),(-20,-10),(-10,0)]
for lo,hi in faixas:
    sub=df[(df.tempo_rel_vfon_s>=lo*60)&(df.tempo_rel_vfon_s<hi*60)]
    if len(sub):
        print(f"  {lo:+3d} a {hi:+3d} min: L_entr={sub.L_entr.mean():5.3f} | "
              f"DET={sub.DET.mean():5.1f} | LAM={sub.LAM.mean():5.1f} | n={len(sub)}")

# --- TESTE DE PREDIÇÃO: longe (-60..-30) vs perto (-30..0) ---
print("\n=== TESTE DE PREDIÇÃO: longe (-60..-30) vs perto (-30..0 min) ===")
longe = df[df.faixa=='longe']; perto = df[df.faixa=='perto']
pred=[]
for feat in FEATURES:
    d=abs(cohen_d(longe[feat].dropna(), perto[feat].dropna()))
    pred.append({'feature':feat,'cohen_d_predicao':round(d,3)})
df_pred=pd.DataFrame(pred).sort_values('cohen_d_predicao',ascending=False)
df_pred.to_csv(os.path.join(DRIVE_TCC,'teste_predicao.csv'), index=False)
print(df_pred.to_string(index=False))
print("\n💾 Salvo: teste_predicao.csv")

d_max = df_pred.cohen_d_predicao.max()
print(f"\n>>> Maior Cohen d na predição: {d_max:.3f}")
if d_max < 0.2:
    print(">>> CONCLUSÃO: nenhuma feature antecipa a FV (d<0.2). O RQA NÃO prediz a FV")
    print("    nos 60 min anteriores. → posicionar o TCC como CARACTERIZAÇÃO/DETECÇÃO.")
elif d_max < 0.5:
    print(">>> Tendência preditiva FRACA. Sinal existe mas é sutil.")
else:
    print(">>> Tendência preditiva RELEVANTE (d>=0.5). Investigar a fundo — pode ser o achado!")

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

df = pd.read_csv(os.path.join(DRIVE_TCC,'manifesto_p10.csv'))
FEATURES = ['RR','DET','L','L_max','DIV','L_entr','LAM','TT','V_max',
            'V_entr','W','W_max','W_entr','DET_RR','LAM_DET']

df['bin'] = (df.tempo_rel_vfon_s // 60 * 60)   # bins de 1 min

fig, axes = plt.subplots(5, 3, figsize=(19, 24))
fig.suptitle('Trajetória das 15 métricas do RQA ao longo do tempo até o colapso (SDDB, p40)',
             fontsize=20, fontweight='bold', y=0.997)

for ax, feat in zip(axes.flat, FEATURES):
    traj = df.groupby('bin')[feat].agg(['mean','std','count']).reset_index()
    traj = traj[traj['count'] >= 3]
    epm = traj['std'] / np.sqrt(traj['count'])
    ax.fill_between(traj.bin/60, traj['mean']-epm, traj['mean']+epm, alpha=0.2, color='tab:blue')
    ax.plot(traj.bin/60, traj['mean'], color='darkblue', lw=2, zorder=3)
    ax.axvline(0, color='red', lw=2, alpha=0.4, zorder=1, label='colapso' if feat==FEATURES[0] else None)
    ax.axvline(-30, color='orange', ls='--', lw=1.2)
    ax.set_title(feat, fontsize=16, fontweight='bold', pad=8)
    ax.set_xlabel('min rel. ao colapso', fontsize=13, fontweight='bold')
    ax.set_ylabel(feat, fontsize=13, fontweight='bold')
    ax.tick_params(labelsize=11)
    ax.grid(True, alpha=0.35, linestyle='--')

# esconde eixos sobrando (15 features, 15 slots — nenhum sobra, mas por segurança)
for ax in axes.flat[len(FEATURES):]:
    ax.set_visible(False)

handles, labels = axes.flat[0].get_legend_handles_labels()
if handles:
    fig.legend(handles, labels, loc='upper right', fontsize=14,
               bbox_to_anchor=(0.995, 0.985))
plt.tight_layout(rect=[0, 0, 1, 0.985])
plt.savefig(os.path.join(IMG_DIR, 'trajetorias_15_features.png'),
            dpi=200, bbox_inches='tight')
plt.show()
print("💾 Salvo: trajetorias_15_features.png")

## Part 8 — Per-Patient Statistics (paired Cohen's d)

In [ ]:
# Um valor por paciente por faixa (média das janelas) -> evita pseudo-replicação
df = pd.read_csv(os.path.join(DRIVE_TCC,'manifesto_p40.csv'))
FEAT = 'L_entr'

por_paciente = (df.groupby(['record','faixa'])[FEAT].mean()
                  .unstack('faixa').dropna())
print(f"=== {FEAT} médio por paciente (longe vs perto do colapso) ===")
print(por_paciente.round(3).to_string())

if len(por_paciente) >= 3 and {'longe','perto'}.issubset(por_paciente.columns):
    try:
        stat, p = wilcoxon(por_paciente['longe'], por_paciente['perto'])
        print(f"\nWilcoxon pareado (longe vs perto): p = {p:.4f}")
        print("✅ Diferença significativa" if p<0.05 else "❌ Não significativa")
    except ValueError as e:
        print(f"Wilcoxon não pôde ser calculado: {e}")
else:
    print("\n⚠️ Poucos pacientes / faixas ausentes para o teste pareado.")

por_paciente.to_csv(os.path.join(DRIVE_TCC,'predicao_por_paciente.csv'))
print("💾 Salvo: predicao_por_paciente.csv")

## Part 9 — HRV / Takens Recurrence Arm

## 11.1. Pré-requisito — anotações auditadas (`.atr`)

O cache da Parte 1 baixa apenas `.hea` e `.dat`. A série NN exige o **`.atr`**, que traz os rótulos
de batimento auditados (`N` sinusal, `V` ventricular, etc.). Sem ele não há como distinguir RR de NN,
e todo o resto perde o sentido.

A célula abaixo baixa os `.atr` que faltarem e, se o módulo `hrv_sddb.py` da orientação estiver no
Drive, o importa como `H` (opcional — o notebook roda sem ele).


In [ ]:
# =============================================================================
#  11.1 · BAIXA OS .atr E (OPCIONAL) IMPORTA O MÓDULO hrv_sddb.py
# =============================================================================
import wfdb

DIR_SDDB_HRV = os.path.join(DRIVE_TCC, 'SDDB_hrv')   # cache dos .hea/.dat/.atr
os.makedirs(DIR_SDDB_HRV, exist_ok=True)

# --- Metadados da SDDB (transcritos da página da base no PhysioNet) ---
#     ritmo de base + disponibilidade de .atr definem quem é elegível para HRV.
SDDB_RITMO = {
    '30':'sinusal','31':'sinusal','32':'marcapasso_int','33':'sinusal','34':'sinusal',
    '35':'fibrilacao_atrial','36':'fibrilacao_atrial','37':'fibrilacao_atrial','38':'sinusal',
    '39':'sinusal','40':'marcapasso_cont','41':'sinusal','42':'sinusal','43':'marcapasso_int',
    '44':'sinusal','45':'sinusal','46':'sinusal','47':'sinusal','48':'sinusal',
    '49':'marcapasso_int','50':'fibrilacao_atrial','51':'marcapasso_int','52':'sinusal',
}
EXCLUIR_RITMO = {'fibrilacao_atrial', 'marcapasso_cont', 'marcapasso_int'}

def baixar_registro_hrv(rec, anotador='atr', pasta=DIR_SDDB_HRV):
    """Garante .hea/.dat/.atr locais. Devolve o caminho-base do registro."""
    base = os.path.join(pasta, rec)
    faltando = [f'{rec}.{ext}' for ext in ('hea', 'dat', anotador)
                if not os.path.exists(os.path.join(pasta, f'{rec}.{ext}'))]
    if faltando:
        try:
            wfdb.dl_files('sddb', pasta, faltando)
        except Exception as e:
            # .atr pode simplesmente não existir para o registro
            print(f"  {rec}: falha ao baixar {faltando} -> {type(e).__name__}")
    return base

def tem_atr(rec, pasta=DIR_SDDB_HRV):
    return os.path.exists(os.path.join(pasta, f'{rec}.atr'))

def vfon_do_header(rec, pasta=DIR_SDDB_HRV):
    """vfon em segundos, lido do comentário #vfon: HH:MM:SS do header."""
    hea = os.path.join(pasta, f'{rec}.hea')
    if not os.path.exists(hea):
        return VFON.get(rec)
    with open(hea, errors='ignore') as f:
        for linha in f:
            if 'vfon' in linha.lower():
                hh, mm, ss = map(int, linha.split(':', 1)[1].strip().split(':'))
                return hh * 3600 + mm * 60 + ss
    return VFON.get(rec)

# --- Baixa tudo e monta a tabela de elegibilidade ---
TODOS_REC = [str(r) for r in wfdb.get_record_list('sddb')]
linhas = []
print("Baixando .hea/.dat/.atr (pula o que já existe)...\n")
for rec in TODOS_REC:
    baixar_registro_hrv(rec)
    ritmo = SDDB_RITMO.get(rec, 'desconhecido')
    vf    = vfon_do_header(rec)
    atr   = tem_atr(rec)
    motivos = []
    if ritmo in EXCLUIR_RITMO: motivos.append(ritmo)
    if not atr:                motivos.append('sem .atr')
    if vf is None:             motivos.append('sem vfon')
    linhas.append({'record': rec, 'ritmo': ritmo, 'tem_atr': atr,
                   'vfon_s': vf, 'elegivel': not motivos,
                   'motivo_exclusao': ', '.join(motivos) or '—'})

META_HRV = pd.DataFrame(linhas)
ELEGIVEIS_HRV = META_HRV.loc[META_HRV.elegivel, 'record'].tolist()

print("\n" + "=" * 78)
print("  ELEGIBILIDADE PARA HRV".center(78))
print("=" * 78)
display(META_HRV.style.set_table_styles(
    [{'selector': 'th', 'props': [('font-weight', 'bold')]}]).hide(axis='index'))
print(f"\n  ELEGIVEIS ({len(ELEGIVEIS_HRV)} de {len(TODOS_REC)}): {ELEGIVEIS_HRV}")
print("\n  Sete de 23 nao e conveniencia: HRV mede modulacao do no sinusal.")
print("  Se o RR vem de conducao AV irregular (FA) ou de um dispositivo (marcapasso),")
print("  SDNN e RMSSD ainda sao calculaveis e sao fisiologicamente sem sentido.")
print("=" * 78)

# --- Módulo da orientação (opcional) ---
H = None
for cand in (DRIVE_TCC, os.path.join(DRIVE_TCC, 'hrv'), '/content', '.'):
    if os.path.exists(os.path.join(cand, 'hrv_sddb.py')):
        sys.path.insert(0, cand)
        try:
            import hrv_sddb as H
            print(f"\n  Modulo hrv_sddb.py importado como H  (de {cand})")
        except Exception as e:
            print(f"\n  hrv_sddb.py encontrado mas nao importou: {type(e).__name__}: {e}")
        break
else:
    print("\n  hrv_sddb.py nao encontrado — o notebook roda sem ele (implementacao propria abaixo).")

## 11.2. Da série RR para a série NN

Esta é a etapa que separa um pipeline de HRV de um contador de batimentos.

**RR ≠ NN.** Um intervalo só é NN se **ambos** os batimentos que o delimitam forem sinusais. Um PVC
encurta o intervalo anterior e alonga o seguinte (pausa compensatória): incluir esse par joga
centenas de ms de variabilidade **não-autonômica** dentro do SDNN. Por isso `guard=1`, que também
descarta os intervalos vizinhos ao ectópico.

Note que isso usa os **rótulos auditados** — informação que nenhum detector automático fornece. Num
pipeline de produção sem anotação, esta etapa vira um classificador de batimentos, e a incerteza dele
se propaga para todos os índices.


In [ ]:
# =============================================================================
#  11.2 · SÉRIE NN A PARTIR DAS ANOTAÇÕES AUDITADAS (.atr)
# =============================================================================

# Rótulos WFDB que são batimentos (o resto é anotação de ritmo/ruído)
SIMB_BATIMENTO = set('NLRejAaJSVEFP/fQ?')
# Rótulos de origem SINUSAL (os únicos que geram intervalo NN válido)
SIMB_SINUSAL   = set('NLRej')

def serie_nn(rec, fs=FS, guard=1, pasta=DIR_SDDB_HRV):
    """
    Constrói a série NN a partir das anotações auditadas.

    Um intervalo entra apenas se AMBOS os batimentos que o delimitam forem sinusais
    e nenhum deles estiver a menos de `guard` batimentos de um ectópico
    (a pausa compensatória contamina tanto quanto o próprio ectópico).

    Devolve (t_s, nn_ms, taxa_aproveitamento):
      t_s   -> instante (s desde o início do registro) do 2º batimento de cada par
      nn_ms -> intervalo NN em milissegundos
    """
    ann = wfdb.rdann(os.path.join(pasta, rec), 'atr')
    simb = np.asarray(ann.symbol)
    amos = np.asarray(ann.sample)

    # 1. mantém só batimentos
    eh_bat = np.array([s in SIMB_BATIMENTO for s in simb])
    simb, amos = simb[eh_bat], amos[eh_bat]
    if len(amos) < 3:
        return np.array([]), np.array([]), 0.0

    # 2. marca sinusais
    ok = np.array([s in SIMB_SINUSAL for s in simb])

    # 3. guard: bloqueia os vizinhos de qualquer não-sinusal
    if guard > 0:
        mau = ~ok
        bloq = mau.copy()
        for g in range(1, guard + 1):
            bloq[g:]  |= mau[:-g]
            bloq[:-g] |= mau[g:]
        ok = ok & ~bloq

    # 4. um intervalo só vale se os DOIS batimentos que o delimitam valem
    t = amos / fs
    rr = np.diff(t) * 1000.0
    valido = ok[:-1] & ok[1:]

    aproveitamento = valido.mean() * 100 if len(valido) else 0.0
    return t[1:][valido], rr[valido], aproveitamento

def limpar_nn(t_s, nn_ms, lo=300, hi=2000):
    """Descarta intervalos fisiologicamente impossíveis (< 300 ms ou > 2000 ms)."""
    m = (nn_ms >= lo) & (nn_ms <= hi)
    return t_s[m], nn_ms[m]

# ------------------------------- DEMONSTRAÇÃO NUM REGISTRO -------------------
REC_DEMO = ELEGIVEIS_HRV[0] if ELEGIVEIS_HRV else '30'
t_nn, nn, aprov = serie_nn(REC_DEMO)
t_nn, nn = limpar_nn(t_nn, nn)
vf_demo = vfon_do_header(REC_DEMO)

print(f"Registro {REC_DEMO}")
print(f"  intervalos NN validos : {len(nn)}")
print(f"  aproveitamento        : {aprov:.1f}% dos intervalos entre batimentos")
print(f"  NN medio              : {nn.mean():.1f} ms  ({60000/nn.mean():.0f} bpm)")
print(f"  SDNN (registro todo)  : {nn.std(ddof=1):.1f} ms")
print(f"  RMSSD (registro todo) : {np.sqrt(np.mean(np.diff(nn)**2)):.1f} ms")
print(f"  vfon                  : {vf_demo} s  ({vf_demo/3600:.2f} h)")

# ------------------------------- TACOGRAMA -----------------------------------
t_rel_min = (t_nn - vf_demo) / 60.0     # minutos relativos ao colapso

fig, ax = plt.subplots(figsize=(15, 4.5))
ax.plot(t_rel_min, nn, color='#1f4e79', lw=0.8, alpha=0.85)
ax.axvline(0, color='crimson', lw=2.5, label='Colapso (vfon)')
ax.axvline(-30, color='darkorange', ls='--', lw=2, label='-30 min')
ax.set_xlim(-90, 5)
ax.set_title(f'Tacograma da serie NN ate o colapso — registro {REC_DEMO}',
             fontsize=17, fontweight='bold', pad=12)
ax.set_xlabel('Minutos relativos ao colapso', fontsize=14, fontweight='bold')
ax.set_ylabel('Intervalo NN (ms)', fontsize=14, fontweight='bold')
ax.legend(loc='upper left', fontsize=13)
ax.grid(True, alpha=0.35, ls='--')
plt.tight_layout()
plt.show()

## 11.3. Reconstrução do espaço de fase (Takens): estimando τ e m

Aqui está a resposta direta à crítica **R1** do parecer ("*espaço de fase não cumprido, m = 1*").

Na amplitude do ECG, `m = 1` era defensável — o sinal já é a observável de interesse. No tacograma,
não é: a série NN é uma projeção unidimensional de uma dinâmica autonômica de dimensão maior, e o
padrão da área é **reconstruir o atrator** antes de medir recorrência.

Dois parâmetros, dois métodos consagrados:

- **τ (atraso)** — primeiro mínimo local da **informação mútua média** (Fraser & Swinney). O primeiro
  mínimo é onde as coordenadas param de ser redundantes sem ainda serem independentes demais.
- **m (dimensão)** — **falsos vizinhos próximos** (FNN, Kennel et al.). Aumenta-se m até que a fração
  de vizinhos que "se separam" ao ganhar uma dimensão caia abaixo de ~1%: o atrator desdobrou.

> **Importante:** τ e m são **estimados por registro**, não chutados. Isso é exatamente o que o
> parecer pede em `[[AUTORES: τ por informação mútua, m por FNN]]`.


In [ ]:
# =============================================================================
#  11.3 · EMBEDDING DE TAKENS — tau por informacao mutua, m por FNN
# =============================================================================
from scipy.spatial import cKDTree

def embed(x, m, tau):
    """Matriz de embedding (N x m) por atrasos: linha i = [x_i, x_{i+tau}, ...]."""
    x = np.asarray(x, dtype=float)
    N = len(x) - (m - 1) * tau
    if N <= 0:
        return np.empty((0, m))
    idx = np.arange(N)[:, None] + np.arange(m)[None, :] * tau
    return x[idx]

def info_mutua(x, max_lag=40, bins=16):
    """Informacao mutua media I(tau) entre x[t] e x[t+tau], via histograma 2D."""
    x = np.asarray(x, dtype=float)
    vals = []
    for lag in range(1, max_lag + 1):
        a, b = x[:-lag], x[lag:]
        c, _, _ = np.histogram2d(a, b, bins=bins)
        pab = c / c.sum()
        pa = pab.sum(axis=1, keepdims=True)
        pb = pab.sum(axis=0, keepdims=True)
        prod = pa @ pb
        nz = (pab > 0) & (prod > 0)
        vals.append(float(np.sum(pab[nz] * np.log(pab[nz] / prod[nz]))))
    return np.array(vals)

def tau_por_info_mutua(x, max_lag=40, bins=16):
    """tau = primeiro MINIMO LOCAL da informacao mutua (Fraser & Swinney)."""
    im = info_mutua(x, max_lag=max_lag, bins=bins)
    for i in range(1, len(im) - 1):
        if im[i] < im[i - 1] and im[i] <= im[i + 1]:
            return i + 1, im
    return int(np.argmin(im)) + 1, im          # fallback: minimo global

def m_por_fnn(x, tau, m_max=10, Rtol=15.0, Atol=2.0, limiar=0.01):
    """
    m = menor dimensao com fracao de falsos vizinhos < `limiar` (Kennel et al.).
    Criterio 1: a distancia extra cresce mais que Rtol vezes a distancia original.
    Criterio 2: a distancia final excede Atol vezes o desvio-padrao da serie.
    """
    x = np.asarray(x, dtype=float)
    sigma = x.std()
    fracoes = []
    for m in range(1, m_max + 1):
        Y = embed(x, m, tau)
        N = len(Y) - tau                       # precisa da coordenada m*tau a frente
        if N < 20:
            break
        arv = cKDTree(Y[:N])
        d, viz = arv.query(Y[:N], k=2)         # k=2: o proprio ponto + vizinho mais proximo
        d1, viz = d[:, 1], viz[:, 1]
        prox_i = x[np.arange(N) + m * tau]
        prox_v = x[viz + m * tau]
        delta  = np.abs(prox_i - prox_v)
        with np.errstate(divide='ignore', invalid='ignore'):
            c1 = np.where(d1 > 0, delta / d1, np.inf) > Rtol
        c2 = (np.sqrt(d1**2 + delta**2) / sigma) > Atol if sigma > 0 else np.zeros(N, bool)
        f = float(np.mean(c1 | c2))
        fracoes.append(f)
        if f < limiar:
            return m, np.array(fracoes)
    return (int(np.argmin(fracoes)) + 1 if fracoes else 2), np.array(fracoes)

# ------------------- ESTIMATIVA NA BASELINE DO REGISTRO DEMO -----------------
# Usa a faixa BASAL (-60 a -30 min) — os parametros descrevem a dinamica NORMAL,
# nao a do colapso. Estimar no colapso seria circular.
mask_basal = (t_rel_min >= -60) & (t_rel_min <= -30)
nn_basal = nn[mask_basal]
print(f"Estimando tau e m na baseline do registro {REC_DEMO} "
      f"({len(nn_basal)} intervalos NN, -60 a -30 min)\n")

TAU_EST, curva_im  = tau_por_info_mutua(nn_basal, max_lag=40)
M_EST,   curva_fnn = m_por_fnn(nn_basal, TAU_EST, m_max=10)

print(f"  tau (1o minimo da informacao mutua) = {TAU_EST} batimentos")
print(f"  m   (fracao de FNN < 1%)            = {M_EST}")
print(f"  -> cada ponto do espaco de fase cobre {(M_EST-1)*TAU_EST} batimentos "
      f"(~{(M_EST-1)*TAU_EST*nn.mean()/1000:.1f} s de sinal)")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

ax1.plot(np.arange(1, len(curva_im) + 1), curva_im, 'o-', color='#1f4e79', ms=5)
ax1.axvline(TAU_EST, color='crimson', ls='--', lw=2.5, label=f'tau escolhido = {TAU_EST}')
ax1.set_title('Informacao mutua media I(tau)', fontsize=16, fontweight='bold', pad=10)
ax1.set_xlabel('Atraso tau (batimentos)', fontsize=14, fontweight='bold')
ax1.set_ylabel('I(tau)  [nats]', fontsize=14, fontweight='bold')
ax1.legend(fontsize=13); ax1.grid(True, alpha=0.35, ls='--')

ax2.plot(np.arange(1, len(curva_fnn) + 1), np.array(curva_fnn) * 100, 'o-',
         color='#7b3294', ms=6)
ax2.axhline(1.0, color='gray', ls=':', lw=2, label='limiar 1%')
ax2.axvline(M_EST, color='crimson', ls='--', lw=2.5, label=f'm escolhido = {M_EST}')
ax2.set_title('Falsos vizinhos proximos (FNN)', fontsize=16, fontweight='bold', pad=10)
ax2.set_xlabel('Dimensao de embedding m', fontsize=14, fontweight='bold')
ax2.set_ylabel('Falsos vizinhos (%)', fontsize=14, fontweight='bold')
ax2.legend(fontsize=13); ax2.grid(True, alpha=0.35, ls='--')

fig.suptitle(f'Reconstrucao do espaco de fase — registro {REC_DEMO} (baseline)',
             fontsize=18, fontweight='bold', y=1.03)
plt.tight_layout()
plt.savefig(os.path.join(IMG_DIR, f'takens_tau_m_{REC_DEMO}.png'), dpi=200, bbox_inches='tight')
plt.show()
print(f"\nSalvo: takens_tau_m_{REC_DEMO}.png")

## 11.4. RQA sobre o tacograma — janela deslizante por batimentos

Agora o núcleo: **o mesmo RQA da Parte 3**, sem alterar uma linha das funções `binarizar` e
`calcular_rqa_completo`. O que muda é apenas **o que entra** — em vez da matriz de distância da
amplitude, a do **atrator reconstruído da série NN**.

```
amplitude do ECG (m=1)  ->  pdist  ->  binarizar  ->  calcular_rqa_completo   [Parte 5]
série NN embarcada (m>1) ->  pdist  ->  binarizar  ->  calcular_rqa_completo   [aqui]
```

**Janela de 256 batimentos, passo de 32.** 256 batimentos ≈ 4–5 min — a escala de curto prazo do
Task Force, e a mesma ordem de grandeza usada por Khazaei et al. (2018), que reivindicam detecção de
morte súbita 6 min antes usando laminaridade do RQA sobre HRV. É por isso que **LAM, L_entr e V_entr**
são as métricas a observar primeiro aqui — não DET e Lmax, que eram as da amplitude.

O **portão de qualidade** (`MIN_APROV`) exige que a janela tenha densidade suficiente de intervalos
NN válidos. A taxa de aprovação ao longo do tempo é registrada e plotada, porque descartar em
silêncio as janelas mais próximas da parada seria esconder o viés, não controlá-lo.


In [ ]:
# =============================================================================
#  11.4 · RQA SOBRE O TACOGRAMA — janela deslizante por batimentos
# =============================================================================

# ---------------------------- Parametros do braco HRV ------------------------
JANELA_BAT   = 256     # batimentos por janela (~4-5 min; padrao curto prazo)
PASSO_BAT    = 32      # deslocamento entre janelas
PERC_HRV     = 10      # percentil de binarizacao (RR-alvo da matriz de recorrencia)
MIN_APROV    = 0.80    # densidade minima de NN validos na janela (portao de qualidade)
MIN_ANTES_HRV= 90      # min antes do vfon a considerar
MIN_POS_HRV  = 5       # min depois do vfon

def rqa_de_serie(x, m, tau, percentil=PERC_HRV, l_min=L_MIN, v_min=V_MIN):
    """RQA de uma serie 1D via embedding de Takens. Reusa as funcoes da Parte 3."""
    Y = embed(np.asarray(x, float), m, tau)
    if len(Y) < 10:
        return None
    D = squareform(pdist(Y, metric='euclidean'))
    rp = binarizar(D, percentil)
    if rp is None:
        return None
    return calcular_rqa_completo(rp, l_min=l_min, v_min=v_min)

def rqa_hrv_do_registro(rec, m=None, tau=None, janela=JANELA_BAT, passo=PASSO_BAT,
                        percentil=PERC_HRV, verbose=True):
    """
    Pipeline completo de um registro:
      .atr -> serie NN -> janelas de N batimentos -> Takens -> RQA (15 metricas)
    Devolve DataFrame com uma linha por janela.
    """
    t_s, nn_ms, _ = serie_nn(rec)
    t_s, nn_ms = limpar_nn(t_s, nn_ms)
    vf = vfon_do_header(rec)
    if vf is None or len(nn_ms) < janela * 2:
        if verbose:
            print(f"  {rec}: dados insuficientes (vfon={vf}, NN={len(nn_ms)}).")
        return pd.DataFrame()

    t_min = (t_s - vf) / 60.0
    faixa_ok = (t_min >= -MIN_ANTES_HRV) & (t_min <= MIN_POS_HRV)
    t_s, nn_ms, t_min = t_s[faixa_ok], nn_ms[faixa_ok], t_min[faixa_ok]
    if len(nn_ms) < janela * 2:
        if verbose:
            print(f"  {rec}: poucos NN na faixa de interesse ({len(nn_ms)}).")
        return pd.DataFrame()

    # tau e m estimados na BASELINE deste registro (nao no colapso)
    if m is None or tau is None:
        base = nn_ms[(t_min >= -60) & (t_min <= -30)]
        if len(base) < 100:
            base = nn_ms[:max(200, len(nn_ms) // 3)]
        tau_r, _ = tau_por_info_mutua(base, max_lag=40)
        m_r,   _ = m_por_fnn(base, tau_r, m_max=8)
    else:
        tau_r, m_r = tau, m

    linhas = []
    for ini in range(0, len(nn_ms) - janela + 1, passo):
        jan_nn = nn_ms[ini:ini + janela]
        jan_t  = t_min[ini:ini + janela]

        # portao de qualidade: a janela precisa ser temporalmente compacta.
        # Se os NN validos estao esparsos, o intervalo coberto estoura o esperado.
        dur_esperada_min = janela * np.median(jan_nn) / 60000.0
        dur_real_min     = jan_t[-1] - jan_t[0]
        aprov = dur_esperada_min / dur_real_min if dur_real_min > 0 else 0.0
        aprovada = aprov >= MIN_APROV

        reg = {'record': rec, 'i_janela': ini // passo,
               't_fim_min': float(jan_t[-1]), 't_ini_min': float(jan_t[0]),
               'tempo_rel_vfon_s': float(jan_t[-1] * 60),
               'aprovacao': round(float(aprov), 3), 'aprovada': bool(aprovada),
               'm': m_r, 'tau': tau_r,
               'NN_medio': float(jan_nn.mean()),
               'SDNN': float(jan_nn.std(ddof=1)),
               'RMSSD': float(np.sqrt(np.mean(np.diff(jan_nn) ** 2)))}

        if aprovada:
            met = rqa_de_serie(jan_nn, m_r, tau_r, percentil)
            if met:
                reg.update(met)
        # faixa, no mesmo vocabulario da Parte 5
        t_fim = reg['t_fim_min']
        reg['faixa'] = 'pos' if t_fim > 0 else ('longe' if t_fim < -30 else 'perto')
        linhas.append(reg)

    df = pd.DataFrame(linhas)
    if verbose:
        n_ok = int(df.aprovada.sum())
        print(f"  {rec}: {len(df)} janelas | {n_ok} aprovadas ({n_ok/len(df)*100:.0f}%) "
              f"| m={m_r}, tau={tau_r}")
    return df

# ------------------------------ RODA NO REGISTRO DEMO ------------------------
print(f"RQA sobre a HRV — registro {REC_DEMO}\n")
df_hrv_demo = rqa_hrv_do_registro(REC_DEMO)
display(df_hrv_demo.head(10).style.set_table_styles(
    [{'selector': 'th', 'props': [('font-weight', 'bold')]}]).hide(axis='index'))

### 11.5. Trajetória das métricas de recorrência da HRV até o colapso

A figura da verdade. Se as curvas forem **planas** até o colapso, o precursor não está no domínio
R-R e temos um **duplo achado negativo** — que ainda assim fecha o contraste de domínio e é honesto.
Se houver **inflexão minutos antes**, onde a amplitude não mostrou nada, a tese vira positiva.

O painel inferior mostra a **taxa de aprovação do portão** ao longo do tempo. Leia os dois juntos:
uma "melhora" da métrica que coincide com queda da cobertura não é fisiologia, é seleção.


In [ ]:
# =============================================================================
#  11.5 · TRAJETORIA DAS METRICAS DE RQA-HRV ATE O COLAPSO
# =============================================================================
FEATS_HRV = ['LAM', 'L_entr', 'V_entr', 'DET', 'TT', 'L_max']   # LAM/entropias primeiro

d = df_hrv_demo[df_hrv_demo.aprovada].copy()

if d.empty or not set(FEATS_HRV).issubset(d.columns):
    print("Nenhuma janela aprovada com metricas. Afrouxe MIN_APROV ou reduza JANELA_BAT.")
else:
    fig = plt.figure(figsize=(17, 13))
    gs  = fig.add_gridspec(4, 3, height_ratios=[1, 1, 1, 0.75], hspace=0.42, wspace=0.26)

    for k, feat in enumerate(FEATS_HRV):
        ax = fig.add_subplot(gs[k // 3, k % 3])
        ax.plot(d.t_fim_min, d[feat], 'o-', color='#1f4e79', ms=4, lw=2, alpha=0.9)
        # banda basal (-60 a -30 min): media +- 2 desvios
        base = d[(d.t_fim_min >= -60) & (d.t_fim_min <= -30)][feat].dropna()
        if len(base) > 3:
            mu, sd = base.mean(), base.std()
            ax.axhspan(mu - 2*sd, mu + 2*sd, color='seagreen', alpha=0.15,
                       label='Banda basal (+/- 2 sd)')
            ax.axhline(mu, color='seagreen', ls=':', lw=2)
        ax.axvline(0,   color='crimson',    lw=2.5, label='Colapso')
        ax.axvline(-30, color='darkorange', ls='--', lw=2, label='-30 min')
        ax.set_title(feat, fontsize=16, fontweight='bold', pad=8)
        ax.set_xlabel('Minutos ate o colapso', fontsize=13, fontweight='bold')
        ax.set_ylabel(feat, fontsize=13, fontweight='bold')
        ax.grid(True, alpha=0.35, ls='--')
        if k == 0:
            ax.legend(fontsize=11, loc='best')

    # ---- painel inferior: cobertura do portao de qualidade ----
    axc = fig.add_subplot(gs[3, :])
    todas = df_hrv_demo.copy()
    axc.fill_between(todas.t_fim_min, 0, todas.aprovacao * 100,
                     color='#7b3294', alpha=0.35, step='mid')
    axc.plot(todas.t_fim_min, todas.aprovacao * 100, color='#7b3294', lw=2)
    axc.axhline(MIN_APROV * 100, color='black', ls='--', lw=2,
                label=f'Portao ({MIN_APROV*100:.0f}%)')
    axc.axvline(0, color='crimson', lw=2.5)
    axc.set_title('Cobertura do portao de qualidade ao longo do tempo '
                  '(viés de seleção dependente do desfecho)',
                  fontsize=15, fontweight='bold', pad=8)
    axc.set_xlabel('Minutos ate o colapso', fontsize=13, fontweight='bold')
    axc.set_ylabel('Densidade de NN (%)', fontsize=13, fontweight='bold')
    axc.legend(fontsize=12); axc.grid(True, alpha=0.35, ls='--')

    fig.suptitle(f'RQA sobre a HRV — registro {REC_DEMO} '
                 f'(m={int(d["m"].iloc[0])}, tau={int(d["tau"].iloc[0])}, '
                 f'janela={JANELA_BAT} batimentos, p{PERC_HRV:02d})',
                 fontsize=19, fontweight='bold', y=0.965)
    plt.savefig(os.path.join(IMG_DIR, f'rqa_hrv_trajetoria_{REC_DEMO}.png'),
                dpi=200, bbox_inches='tight')
    plt.show()
    print(f"Salvo: rqa_hrv_trajetoria_{REC_DEMO}.png")

## 11.6. Lote — todos os registros elegíveis

Roda o pipeline nos 7 elegíveis e salva `manifesto_hrv_rqa.csv` em `TCC/csvs`, no **mesmo formato**
dos manifestos da amplitude (colunas `record`, `tempo_rel_vfon_s`, `faixa` + as 15 métricas). Isso é
proposital: permite que toda a maquinaria de análise das Partes 6–8 (Cohen's d, Z-score, alerta
precoce, consenso MAD, Wilcoxon pareado) seja aplicada ao braço HRV **sem reescrever nada**.


In [ ]:
# =============================================================================
#  11.6 · LOTE — RQA-HRV EM TODOS OS ELEGIVEIS -> manifesto_hrv_rqa.csv
# =============================================================================
from tqdm.auto import tqdm

RODAR_LOTE_HRV = True          # coloque False para so recarregar o CSV ja salvo
SAIDA_HRV = os.path.join(CSV_DIR, 'manifesto_hrv_rqa.csv')

if RODAR_LOTE_HRV:
    partes = []
    for rec in tqdm(ELEGIVEIS_HRV, desc='RQA-HRV', unit='paciente'):
        try:
            partes.append(rqa_hrv_do_registro(rec, verbose=True))
        except Exception as e:
            print(f"  {rec}: ERRO {type(e).__name__}: {e}")
    df_hrv = pd.concat([p for p in partes if not p.empty], ignore_index=True) \
             if partes else pd.DataFrame()
    if not df_hrv.empty:
        os.makedirs(CSV_DIR, exist_ok=True)
        df_hrv.to_csv(SAIDA_HRV, index=False)
        CSV['manifesto_hrv_rqa'] = df_hrv
        print(f"\nSalvo: {SAIDA_HRV}  ({len(df_hrv)} janelas)")
elif os.path.exists(SAIDA_HRV):
    df_hrv = pd.read_csv(SAIDA_HRV)
    print(f"Recarregado do Drive: {len(df_hrv)} janelas")
else:
    df_hrv = pd.DataFrame()
    print("Nada a carregar — rode com RODAR_LOTE_HRV = True.")

if not df_hrv.empty:
    print("\n" + "=" * 78)
    print("  COBERTURA POR PACIENTE".center(78))
    print("=" * 78)
    resumo = (df_hrv.groupby('record')
              .agg(janelas=('i_janela', 'count'),
                   aprovadas=('aprovada', 'sum'),
                   m=('m', 'first'), tau=('tau', 'first'),
                   NN_medio=('NN_medio', 'mean'))
              .assign(aprovacao_pct=lambda x: (x.aprovadas / x.janelas * 100).round(1))
              .reset_index())
    display(resumo.style.set_table_styles(
        [{'selector': 'th', 'props': [('font-weight', 'bold')]}]).hide(axis='index'))

## 11.7. O contraste de domínio — amplitude vs. HRV

A figura e a tabela que **são a contribuição do paper**. Mesma pergunta preditiva da Parte 7
(`longe` −60/−30 min vs. `perto` −30/0 min), mesma régua (Cohen's d), **por paciente** para evitar
pseudo-replicação. Só muda o domínio do sinal.

Três leituras possíveis:

| Resultado | Interpretação | Destino |
|---|---|---|
| Amplitude ≈ 0 **e** HRV ≈ 0 | Duplo achado negativo; o precursor não está em nenhum dos dois domínios nesta base | SBEB, com tese honesta |
| Amplitude ≈ 0 **e** HRV > 0,5 | **Contraste de domínio fechado** — o precursor é autonômico, invisível à amplitude | IEEE Xplore |
| Ambos altos | Revisar: provavelmente há vazamento de janelas pós-colapso | Investigar antes de reportar |


In [ ]:
# =============================================================================
#  11.7 · CONTRASTE DE DOMINIO — amplitude (Parte 5) vs. HRV (Parte 11)
# =============================================================================

def d_por_paciente(df, features, col_faixa='faixa'):
    """
    Cohen's d longe vs perto com UM valor por paciente por faixa.
    Reescrito para NÃO usar .unstack() e evitar o Pandas4Warning.
    """
    saida = []
    for feat in features:
        if feat not in df.columns:
            continue

        # 1. Filtra e calcula a média por paciente e faixa
        df_filt = df[df[col_faixa].isin(['longe', 'perto'])]
        agg = df_filt.groupby(['record', col_faixa])[feat].mean().reset_index()

        # 2. Mantém apenas os pacientes que têm AS DUAS faixas (longe E perto)
        counts = agg.groupby('record')[col_faixa].nunique()
        pacientes_completos = counts[counts == 2].index

        if len(pacientes_completos) < 3:
            continue

        # 3. Separa os arrays a (longe) e b (perto) garantindo a mesma ordem
        validos = agg[agg['record'].isin(pacientes_completos)].sort_values('record')
        a = validos[validos[col_faixa] == 'longe'][feat].values
        b = validos[validos[col_faixa] == 'perto'][feat].values

        # 4. Cálculo do Desvio Padrão Agrupado (Pooled) e d de Cohen
        sp = np.sqrt(((len(a)-1)*a.std(ddof=1)**2 + (len(b)-1)*b.std(ddof=1)**2)
                     / (len(a)+len(b)-2))

        saida.append({'feature': feat, 'n_pacientes': len(pacientes_completos),
                      'cohen_d': round(abs((a.mean()-b.mean())/sp), 3) if sp > 0 else np.nan,
                      'media_longe': round(a.mean(), 4),
                      'media_perto': round(b.mean(), 4)})

    return pd.DataFrame(saida).sort_values('cohen_d', ascending=False)

# ---- Braco AMPLITUDE (manifesto ja salvo no Drive) ----
df_amp = MANIFESTOS.get(10, MANIFESTOS.get(40))
d_amp = d_por_paciente(df_amp, FEATURES) if df_amp is not None else pd.DataFrame()

# ---- Braco HRV ----
d_hrv = d_por_paciente(df_hrv[df_hrv.aprovada], FEATURES) if not df_hrv.empty else pd.DataFrame()

comp = (d_amp[['feature', 'cohen_d']].rename(columns={'cohen_d': 'd_AMPLITUDE'})
        .merge(d_hrv[['feature', 'cohen_d']].rename(columns={'cohen_d': 'd_HRV'}),
               on='feature', how='outer')) if not d_amp.empty and not d_hrv.empty else pd.DataFrame()

if not comp.empty:
    comp = comp.sort_values('d_HRV', ascending=False)
    print("=" * 78)
    print("  COHEN'S D POR PACIENTE — longe (-60/-30) vs perto (-30/0)".center(78))
    print("=" * 78)
    display(comp.style.background_gradient(subset=['d_AMPLITUDE', 'd_HRV'], cmap='Reds')
                .format({'d_AMPLITUDE': '{:.3f}', 'd_HRV': '{:.3f}'})
                .set_table_styles([{'selector': 'th',
                                    'props': [('font-weight', 'bold')]}])
                .hide(axis='index'))

    fig, ax = plt.subplots(figsize=(15, 7))
    y = np.arange(len(comp)); h = 0.38
    ax.barh(y + h/2, comp.d_AMPLITUDE, h, label='Amplitude do ECG (m = 1)',
            color='#9aa5b1', edgecolor='#333333', linewidth=1.2)
    ax.barh(y - h/2, comp.d_HRV, h, label=f'HRV / tacograma (Takens, m > 1)',
            color='#1f4e79', edgecolor='#333333', linewidth=1.2)
    ax.axvline(0.2, color='darkorange', ls='--', lw=2.5, label="d = 0,2 (efeito pequeno)")
    ax.axvline(0.5, color='crimson',    ls='--', lw=2.5, label="d = 0,5 (efeito medio)")
    ax.set_yticks(y); ax.set_yticklabels(comp.feature, fontsize=13, fontweight='bold')
    ax.set_xlabel("Cohen's d (por paciente) — longe vs perto do colapso",
                  fontsize=15, fontweight='bold')
    ax.set_title('CONTRASTE DE DOMINIO: a assinatura precursora esta na amplitude ou na HRV?',
                 fontsize=18, fontweight='bold', pad=14)
    ax.legend(fontsize=13, loc='lower right')
    ax.grid(True, axis='x', alpha=0.35, ls='--')
    plt.tight_layout()
    plt.savefig(os.path.join(IMG_DIR, 'contraste_dominio_amplitude_vs_hrv.png'),
                dpi=200, bbox_inches='tight')
    plt.show()

    dmax_amp = comp.d_AMPLITUDE.max(); dmax_hrv = comp.d_HRV.max()
    print("\n" + "=" * 78)
    print(f"  Maior d na AMPLITUDE : {dmax_amp:.3f}")
    print(f"  Maior d na HRV       : {dmax_hrv:.3f}   "
          f"(feature: {comp.iloc[0].feature})")
    print("=" * 78)
    if dmax_hrv >= 0.5 and dmax_amp < 0.2:
        print("  >> CONTRASTE DE DOMINIO FECHADO. O precursor e autonomico, invisivel a")
        print("     amplitude. Este e o resultado que sustenta a submissao ao IEEE Xplore.")
    elif dmax_hrv < 0.2:
        print("  >> DUPLO ACHADO NEGATIVO. O precursor nao aparece em nenhum dos dominios")
        print("     nesta base. Resultado honesto e reportavel — posicionar para o SBEB.")
    else:
        print("  >> TENDENCIA FRACA na HRV. Antes de reportar: aumente n, teste outros")
        print("     percentis e confirme que nenhuma janela 'pos' vazou para 'perto'.")
    print("     Lembre: n = 7. Qualquer numero aqui e DESCRITIVO, nao inferencial.")
else:
    print("Rode a Parte 11.6 (e garanta que os manifestos da amplitude estao carregados).")

## 11.8. Braço A — HRV clássica (SDNN / RMSSD) como controle

Complementar e barato: a trajetória dos índices **lineares** de HRV até a parada. Serve de sanidade
para o braço RQA — se SDNN e RMSSD já derivam sozinhos, parte do sinal do RQA-HRV pode ser apenas
isso, e não estrutura de recorrência.

> **Não leia causalidade nestas curvas.** Tempo-até-FV está confundido com ciclo circadiano por
> construção, a medicação é majoritariamente desconhecida (digoxina e quinidina aparecem em vários
> sujeitos — ambas afetam HRV diretamente) e o portão de qualidade correlaciona com o desfecho.
> Para atacar isso de verdade, rode `HRV_SDDB_controles.ipynb`: pseudo-evento em base sem evento,
> sensibilidade ao portão, `gap_handling` e o confundidor da grade de amostragem.


In [ ]:
# =============================================================================
#  11.8 · BRACO A — trajetoria dos indices lineares de HRV (SDNN / RMSSD)
# =============================================================================
if df_hrv.empty:
    print("Rode a Parte 11.6 primeiro.")
else:
    d = df_hrv[df_hrv.aprovada].copy()
    fig, axes = plt.subplots(1, 3, figsize=(18, 5.2))
    painel = [('NN_medio', 'NN medio (ms)',  '#1f4e79'),
              ('SDNN',     'SDNN (ms)',      '#7b3294'),
              ('RMSSD',    'RMSSD (ms)',     '#c1272d')]

    for ax, (col, rot, cor) in zip(axes, painel):
        for rec, g in d.groupby('record'):
            ax.plot(g.t_fim_min, g[col], lw=1.4, alpha=0.55, label=f'reg {rec}')
        med = d.groupby(d.t_fim_min.round(0))[col].median()
        ax.plot(med.index, med.values, color=cor, lw=3.5, zorder=5, label='mediana')
        ax.axvline(0, color='crimson', lw=2.5)
        ax.axvline(-30, color='darkorange', ls='--', lw=2)
        ax.set_title(rot, fontsize=16, fontweight='bold', pad=8)
        ax.set_xlabel('Minutos ate o colapso', fontsize=13, fontweight='bold')
        ax.set_ylabel(rot, fontsize=13, fontweight='bold')
        ax.grid(True, alpha=0.35, ls='--')
    axes[-1].legend(fontsize=10, ncol=2, loc='best')

    fig.suptitle('Braco A — indices lineares de HRV ate o colapso (todos os elegiveis)',
                 fontsize=18, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(os.path.join(IMG_DIR, 'hrv_classica_trajetoria.png'),
                dpi=200, bbox_inches='tight')
    plt.show()

    print("\nATENCAO — o viés de quantizacao é correlacionado com o desfecho:")
    print("  A grade de 250 Hz injeta ruido de ~1,63 ms no RR (sigma_dRR = delta/raiz(2)).")
    print("  Ele soma em quadratura: RMSSD_obs = raiz(RMSSD_real^2 + sigma^2).")
    print("  Logo o vies e ~0,3% quando o RMSSD real e 60 ms e ~30% quando e 5 ms —")
    print("  ou seja, MAXIMO exatamente nos pacientes com falencia autonomica.")
    print("  Uma 'queda de RMSSD' perto da parada pode ser o conversor A/D, nao o paciente.")

## Part 10 — Reviewer Analyses (R-4, R-5, gate, bootstrap)

### 10.1 SDDB demographics/clinical table

In [ ]:
# Dicionário contendo os dados demográficos e clínicos da base SDDB
sddb_patients = {
    "30": {"gender": "Male",    "age": "43",      "history": "Unknown", "medication": "Unknown", "rhythm": "Sinus"},
    "31": {"gender": "Female",  "age": "72",      "history": "Heart failure", "medication": "digoxin; quinidine gluconate", "rhythm": "Sinus"},
    "32": {"gender": "Unknown", "age": "62",      "history": "Coronary bypass grafting; history of arrhythmia", "medication": "Procan SR; beta-blocker", "rhythm": "Sinus with intermittent demand ventricular pacing; CPR at time of cardiac arrest"},
    "33": {"gender": "Female",  "age": "30",      "history": "Unknown", "medication": "Unknown", "rhythm": "Sinus"},
    "34": {"gender": "Male",    "age": "34",      "history": "Unknown", "medication": "Unknown", "rhythm": "Sinus"},
    "35": {"gender": "Female",  "age": "72",      "history": "Mitral valve replacement", "medication": "digoxin", "rhythm": "Atrial fibrillation"},
    "36": {"gender": "Male",    "age": "75",      "history": "Cardiac surgery", "medication": "digoxin; quinidine", "rhythm": "Atrial fibrillation"},
    "37": {"gender": "Female",  "age": "89",      "history": "Unknown", "medication": "Unknown", "rhythm": "Atrial fibrillation"},
    "38": {"gender": "Unknown", "age": "Unknown", "history": "Unknown", "medication": "Unknown", "rhythm": "Sinus"},
    "39": {"gender": "Male",    "age": "66",      "history": "Acute myelogenous leukemia", "medication": "digoxin; quinidine", "rhythm": "Sinus"},
    "40": {"gender": "Male",    "age": "79",      "history": "Unknown", "medication": "Unknown", "rhythm": "Paced"},
    "41": {"gender": "Male",    "age": "Unknown", "history": "Unknown", "medication": "Unknown", "rhythm": "Sinus"},
    "42": {"gender": "Male",    "age": "17",      "history": "Hypertrophic cardiomyopathy; positive family history of sudden death", "medication": "Unknown", "rhythm": "Sinus"},
    "43": {"gender": "Male",    "age": "35",      "history": "Coronary artery disease", "medication": "Unknown", "rhythm": "Intermittent ventricular pacing"},
    "44": {"gender": "Male",    "age": "Unknown", "history": "Unknown", "medication": "Unknown", "rhythm": "Sinus"},
    "45": {"gender": "Male",    "age": "68",      "history": "History of ventricular ectopy", "medication": "digoxin; quinidine gluconate", "rhythm": "Sinus"},
    "46": {"gender": "Female",  "age": "Unknown", "history": "Unknown", "medication": "Unknown", "rhythm": "Sinus"},
    "47": {"gender": "Male",    "age": "34",      "history": "Unknown", "medication": "Unknown", "rhythm": "Sinus"},
    "48": {"gender": "Male",    "age": "80",      "history": "Unknown", "medication": "Unknown", "rhythm": "Sinus"},
    "49": {"gender": "Male",    "age": "73",      "history": "Coronary artery s/p myocardial infarction; history of ventricular tachycardia", "medication": "Unknown", "rhythm": "Sinus with intermittent pacing"},
    "50": {"gender": "Female",  "age": "68",      "history": "Coronary artery bypass graft; mitral valve replacement", "medication": "digoxin; quinidine; propranolol; potassium; diuretics", "rhythm": "Atrial fibrillation"},
    "51": {"gender": "Female",  "age": "67",      "history": "Unknown", "medication": "Unknown", "rhythm": "Sinus with intermittent pacing"},
    "52": {"gender": "Female",  "age": "82",      "history": "Heart failure", "medication": "None listed", "rhythm": "Sinus"}
}

# ------------------------------------------------------------------
# EXTRAÇÃO AUTOMÁTICA DOS DADOS PARA O BRAÇO DE VFC (7 PACIENTES)
# ------------------------------------------------------------------
import numpy as np

# Os 7 pacientes elegíveis do braço VFC (sinusais, com anotação auditada)
braco_vfc_ids = ['30', '31', '34', '41', '45', '46', '52']

idades = []
sexos = {"Male": 0, "Female": 0, "Unknown": 0}
ritmos = set()

for pid in braco_vfc_ids:
    paciente = sddb_patients[pid]

    # Sexo
    sexos[paciente["gender"]] += 1

    # Ritmo
    ritmos.add(paciente["rhythm"])

    # Idade (ignora 'Unknown' para a mediana)
    if paciente["age"] != "Unknown":
        idades.append(int(paciente["age"]))

mediana_idade = np.median(idades) if idades else "N/A"

print("=" * 60)
print("  DADOS PARA PREENCHER NO MANUSCRITO (SEÇÃO II-A)")
print("=" * 60)
print(f"Registros do Braço VFC: {braco_vfc_ids}")
print(f"Idade Mediana: {mediana_idade:.0f} anos (dados disponíveis para {len(idades)} de {len(braco_vfc_ids)} pacientes)")
print(f"Distribuição por Sexo: {sexos['Male']} Homens, {sexos['Female']} Mulheres")
print(f"Ritmo de Base: {', '.join(ritmos)}")
print("=" * 60)

### 10.2 R-5 — rhythm in the [-30 s, 0) window

In [ ]:
# -*- coding: utf-8 -*-
import os
import glob
import pandas as pd
import wfdb

# ---------------------------------------------------------------------------
# AJUSTE AQUI: pasta onde estão os .hea/.dat/.ari da SDDB
# ---------------------------------------------------------------------------
DATA_DIR = "/content/drive/MyDrive/TCC/SDDB_hrv"   # <-- troque se necessário
FS = 250                                            # Hz (SDDB)
JANELA_S = 30                                       # últimos 30 s antes do vfon

# Rótulos de ritmo (sem o parêntese inicial)
SINUSAL = {"N", "NSR"}
TAQUI_V = {"VT", "VFL"}          # taquicardia/flutter ventricular
FIB_V   = {"VF", "VFIB"}         # fibrilação já instalada
OUTROS_CONHECIDOS = {"AFIB", "PM", "B", "BII", "SBR", "SVTA", "NOD",
                     "VER", "HGEA", "ASYS", "NOISE", "BI"}

def limpar_rotulo(aux):
    if aux is None:
        return None
    s = str(aux).replace("\x00", "").strip()
    if not s:
        return None
    if s.startswith("("):
        s = s[1:]
    return s.strip() or None

def vfon_do_header(rec_base):
    hea = rec_base + ".hea"
    if not os.path.exists(hea):
        return None
    with open(hea, errors="ignore") as f:
        for linha in f:
            if "vfon" in linha.lower():
                try:
                    hh, mm, ss = map(int, linha.split(":", 1)[1].strip().split(":"))
                    return (hh * 3600 + mm * 60 + ss) * FS
                except Exception:
                    pass
    return None

def ritmo_no_intervalo(rec_base, ini_amostra, fim_amostra):
    ritmos = []
    # Tenta carregar .ari primeiro (todos têm), depois .atr
    for ext in ["ari", "atr"]:
        try:
            ann = wfdb.rdann(rec_base, ext)
            if ann.aux_note is not None:
                for amostra, aux in zip(ann.sample, ann.aux_note):
                    rot = limpar_rotulo(aux)
                    if rot:
                        ritmos.append((int(amostra), rot))
            if ritmos:
                break # Achou anotações, não precisa ler a outra extensão
        except FileNotFoundError:
            continue

    if not ritmos:
        return set(), "sem anotacao de ritmo"

    vigentes = []
    herdado = None
    for amostra, rot in ritmos:
        if amostra <= ini_amostra:
            herdado = rot
        elif ini_amostra < amostra <= fim_amostra:
            vigentes.append(rot)

    if herdado is not None:
        vigentes.insert(0, herdado)

    return set(vigentes), ";".join(dict.fromkeys(vigentes)) or "(nenhum no intervalo)"

def classificar(conj):
    if conj & TAQUI_V:
        return "TV"
    if conj & FIB_V:
        return "FV_ja"
    if conj & SINUSAL and not (conj & (TAQUI_V | FIB_V)):
        return "sinusal"
    if conj & {"AFIB"}:
        return "FA"
    if conj & {"PM"}:
        return "marcapasso"
    if not conj:
        return "indefinido"
    return "outro"

def descobrir_registros(pasta):
    bases = set()
    for hea in glob.glob(os.path.join(pasta, "**", "*.hea"), recursive=True):
        bases.add(os.path.splitext(hea)[0])
    return sorted(bases, key=lambda p: os.path.basename(p))

def main():
    registros = descobrir_registros(DATA_DIR)
    if not registros:
        print(f"Nenhum .hea encontrado em {DATA_DIR}.")
        return

    linhas = []
    for base in registros:
        rec = os.path.basename(base)
        vfon = vfon_do_header(base)
        if vfon is None:
            linhas.append({"record": rec, "vfon_s": None, "categoria": "sem_vfon", "ritmos_no_intervalo": "-"})
            continue

        ini = max(0, vfon - JANELA_S * FS)
        fim = vfon

        conj, detalhe = ritmo_no_intervalo(base, ini, fim)

        linhas.append({
            "record": rec,
            "vfon_s": vfon // FS,
            "categoria": classificar(conj),
            "ritmos_no_intervalo": detalhe,
        })

    df = pd.DataFrame(linhas).sort_values("record").reset_index(drop=True)

    print("=" * 74)
    print("  R-5 — RITMO EM [-30 s, 0) POR REGISTRO (LENDO .ARI e .ATR)  ".center(74))
    print("=" * 74)
    print(df.to_string(index=False))

    n_sinusal = int((df.categoria == "sinusal").sum())
    n_tv = int((df.categoria == "TV").sum())
    n_fv = int((df.categoria == "FV_ja").sum())
    n_val = int(df.categoria.isin(["sinusal", "TV", "FV_ja", "FA", "marcapasso", "outro"]).sum())

    print("\n" + "=" * 74)
    print("  VEREDITO".center(74))
    print("=" * 74)
    print(f"  Registros avaliados        : {n_val}")
    print(f"  Sinusal em [-30 s, 0)      : {n_sinusal}")
    print(f"  Taquicardia ventricular    : {n_tv}")
    print(f"  Já em FV nesse intervalo   : {n_fv}")

    if n_tv + n_fv > n_sinusal and n_val > 0:
        print("\n  >> A MAIORIA NÃO está em ritmo sinusal nos últimos 30 s.")
        print("     O objeto descrito como 'estado pré-fibrilatório' é, na verdade,")
        print("     TAQUICARDIA VENTRICULAR. O texto do artigo precisa focar na transição VT->FV.")
    elif n_val > 0:
        print("\n  >> A maioria ESTÁ em ritmo sinusal até perto do evento.")
        print("     O enquadramento 'pré-fibrilatório' se sustenta!")

if __name__ == "__main__":
    main()

### 10.3 R-4 — amplitude contrast restricted to the same 7 patients

In [ ]:
# -*- coding: utf-8 -*-
"""
R-4 (bloqueante do parecer CBEB) — Controle do contraste de domínio
====================================================================
Objeção de R2/F2.1: o contraste amplitude vs. VFC compara 20 pacientes
(amplitude) contra 7 (VFC). A diferença de Cohen's d pode vir da amostra,
não do domínio. Este script ELIMINA esse confundimento recomputando o d
far/near do braço de AMPLITUDE restrito EXATAMENTE aos 7 pacientes do
braço de VFC.

Se o d de amplitude nesses 7 permanecer ≈ 0,19 (como nos 20), o contraste
ganha uma perna real: mesma amostra, mesma pergunta, domínios diferentes.
Se subir muito, você precisa saber disso ANTES do revisor.

NÃO reprocessa sinal — só filtra os manifests CSV que você já tem.
Reusa a MESMA função d_per_patient da célula 111 (faixa longe/perto, por record).
"""

import os
import numpy as np
import pandas as pd

# ---------------------------------------------------------------------------
# Reutiliza csv_path e FEATURES se já existirem no notebook; senão, define.
# ---------------------------------------------------------------------------
try:
    csv_path
except NameError:
    DRIVE_TCC = globals().get("DRIVE_TCC", "/content/drive/MyDrive/TCC")
    CSV_DIR = globals().get("CSV_DIR", os.path.join(DRIVE_TCC, "csvs"))
    def csv_path(nome):
        if not nome.endswith(".csv"):
            nome += ".csv"
        for base in (CSV_DIR, DRIVE_TCC):
            p = os.path.join(base, nome)
            if os.path.exists(p):
                return p
        return os.path.join(CSV_DIR, nome)

FEATURES = globals().get("FEATURES", ['RR','DET','L','L_max','DIV','L_entr','LAM',
        'TT','V_max','V_entr','W','W_max','W_entr','DET_RR','LAM_DET'])

P_AMP = 10   # mesmo manifesto de amplitude usado na Fig. 4 (troque p/ 40 se for o caso)


def cohen_d_pareado(far, near):
    """
    Cohen's d PAREADO (d_z): far e near são medidos no MESMO paciente.
    d_z = média(diferença) / desvio-padrão(diferença).
    É o estimador correto para o desenho far/near (R2/F2.2 pede declarar isto).
    """
    dif = np.asarray(far) - np.asarray(near)
    if len(dif) < 2 or dif.std(ddof=1) == 0:
        return np.nan
    return dif.mean() / dif.std(ddof=1)


def d_por_paciente(df_in, features, col_band="faixa", registros=None, pareado=True):
    """
    Cohen's d (far vs near), UM valor por paciente por faixa.
    - registros: se fornecido, restringe a esse subconjunto (o filtro do R-4).
    - pareado=True  -> d_z (recomendado; mesmo paciente nas duas faixas)
      pareado=False -> d_s (entre grupos), como estava na célula 111.
    """
    rows = []
    for feat in features:
        if feat not in df_in.columns:
            continue
        sub = df_in[df_in[col_band].isin(["longe", "perto"])].copy()
        if registros is not None:
            sub = sub[sub["record"].astype(str).isin([str(r) for r in registros])]
        agg = sub.groupby(["record", col_band])[feat].mean().reset_index()
        counts = agg.groupby("record")[col_band].nunique()
        completos = counts[counts == 2].index         # tem far E near
        if len(completos) < 3:
            continue
        ok = agg[agg["record"].isin(completos)].sort_values("record")
        far = ok[ok[col_band] == "longe"][feat].values
        near = ok[ok[col_band] == "perto"][feat].values
        if pareado:
            d = cohen_d_pareado(far, near)
        else:
            sp = np.sqrt(((len(far)-1)*far.std(ddof=1)**2 + (len(near)-1)*near.std(ddof=1)**2)
                         / (len(far)+len(near)-2))
            d = (far.mean()-near.mean())/sp if sp > 0 else np.nan
        rows.append({"feature": feat, "n_patients": len(completos),
                     "cohen_d": round(abs(d), 3) if np.isfinite(d) else np.nan})
    return pd.DataFrame(rows).sort_values("cohen_d", ascending=False).reset_index(drop=True)


# ---------------------------------------------------------------------------
# EXECUÇÃO
# ---------------------------------------------------------------------------
def main():
    # 1. carrega os dois manifests
    df_amp = pd.read_csv(csv_path(f"manifesto_p{P_AMP:02d}.csv"))
    df_hrv = pd.read_csv(csv_path("manifesto_hrv_rqa.csv"))
    if "aprovada" in df_hrv.columns:
        df_hrv = df_hrv[df_hrv.aprovada]

    # 2. descobre os 7 registros do braço de VFC (os que aparecem no manifesto HRV)
    sete = sorted(df_hrv["record"].astype(str).unique(), key=lambda x: int(x))
    print("=" * 72)
    print(f"  Registros do braço de VFC (n={len(sete)}): {sete}")
    print("=" * 72)

    # 3. os três d far/near:
    #    (a) amplitude nos 20 (como no paper)   (b) amplitude nos MESMOS 7   (c) VFC nos 7
    d_amp20_z = d_por_paciente(df_amp, FEATURES, pareado=True)
    d_amp7_z  = d_por_paciente(df_amp, FEATURES, registros=sete, pareado=True)
    d_hrv7_z  = d_por_paciente(df_hrv, FEATURES, pareado=True)
    # versão d_s (entre grupos) só para amplitude-7, para comparar com o d do paper
    d_amp7_s  = d_por_paciente(df_amp, FEATURES, registros=sete, pareado=False)

    comp = (d_amp20_z[["feature", "cohen_d"]].rename(columns={"cohen_d": "AMP_20_dz"})
            .merge(d_amp7_z[["feature", "cohen_d"]].rename(columns={"cohen_d": "AMP_7_dz"}), on="feature", how="outer")
            .merge(d_amp7_s[["feature", "cohen_d"]].rename(columns={"cohen_d": "AMP_7_ds"}), on="feature", how="outer")
            .merge(d_hrv7_z[["feature", "cohen_d"]].rename(columns={"cohen_d": "HRV_7_dz"}), on="feature", how="outer"))
    comp = comp.sort_values("HRV_7_dz", ascending=False).reset_index(drop=True)

    print("\n  Cohen's d por paciente — far vs near")
    print("  AMP_20 = amplitude, 20 pac | AMP_7 = amplitude nos 7 do braço VFC | HRV_7 = VFC")
    print("  dz = pareado (recomendado) · ds = entre grupos (como na Fig. 4 atual)\n")
    print(comp.to_string(index=False))

    amp20 = d_amp20_z.cohen_d.max()
    amp7  = d_amp7_z.cohen_d.max()
    hrv7  = d_hrv7_z.cohen_d.max()
    print("\n" + "=" * 72)
    print("  VEREDITO (maior d de cada braço, pareado d_z)".center(72))
    print("=" * 72)
    print(f"  Amplitude, 20 pacientes : {amp20:.3f}")
    print(f"  Amplitude, MESMOS 7     : {amp7:.3f}   <-- o número que o R-4 pede")
    print(f"  VFC, 7 pacientes        : {hrv7:.3f}")
    if amp7 < 0.2 <= hrv7:
        print("\n  >> CONTRASTE SOBREVIVE ao controle de amostra: mesmos 7 pacientes,")
        print("     amplitude continua sem separar (d<0,2) e VFC separa. Esta é a")
        print("     frase forte para responder R2/F2.1 no texto.")
    elif amp7 >= 0.2:
        print("\n  >> ATENÇÃO: nos mesmos 7, a amplitude também separa (d>=0,2). Parte do")
        print("     contraste vinha da amostra, não do domínio. O texto precisa dizer isto")
        print("     honestamente — e o enquadramento do título precisa ser suavizado.")
    print("=" * 72)

    saida = os.path.join(os.path.dirname(csv_path("x.csv")), "R4_contraste_mesmos7.csv")
    try:
        comp.to_csv(saida, index=False)
        print(f"\nSalvo: {saida}")
    except Exception as e:
        print(f"\n(Não consegui salvar CSV: {e})")

    return comp


if __name__ == "__main__":
    main()

### 10.4 Quality-gate coverage over time

In [ ]:
import pandas as pd
import os

# Caminho do manifesto de VFC (o mesmo usado no script R-4)
caminho_hrv = "/content/drive/MyDrive/TCC/csvs/manifesto_hrv_rqa.csv"

if os.path.exists(caminho_hrv):
    df_hrv = pd.read_csv(caminho_hrv)

    print("=" * 60)
    print("  COBERTURA DO GATE DE VFC NO TEMPO".center(60))
    print("=" * 60)

    if 'faixa' in df_hrv.columns and 'aprovada' in df_hrv.columns:
        # Filtra apenas as fases que importam
        df_longe = df_hrv[df_hrv['faixa'] == 'longe']
        df_perto = df_hrv[df_hrv['faixa'] == 'perto']

        # Calcula as taxas
        taxa_longe = (df_longe['aprovada'].sum() / len(df_longe)) * 100 if len(df_longe) > 0 else 0
        taxa_perto = (df_perto['aprovada'].sum() / len(df_perto)) * 100 if len(df_perto) > 0 else 0

        print(f"  Fase FAR (-60 a -30 min) : {taxa_longe:.1f}% das janelas aprovadas")
        print(f"  Fase NEAR (-30 a 0 min)  : {taxa_perto:.1f}% das janelas aprovadas")
        print("=" * 60)

        if taxa_perto < taxa_longe:
            print("\n  >> Confirmado: A ectopia aumenta perto do colapso e o")
            print("     gate de qualidade rejeita mais janelas na fase NEAR.")
    else:
        print("As colunas 'faixa' ou 'aprovada' não foram encontradas.")
else:
    print("Arquivo manifesto_hrv_rqa.csv não encontrado no caminho.")

### 10.5 Lead-time test (mu+3sigma) and bootstrap 95% CI (real data)

In [ ]:
# =============================================================================
# SCRIPT DE REVISÃO FINAL: R-7 (Lead Time) e Bootstrap (IC 95%) REAL
# =============================================================================
import numpy as np
import pandas as pd
from scipy.stats import bootstrap
import os

# ---------------------------------------------------------------------------
# PREPARAÇÃO DOS DADOS
# ---------------------------------------------------------------------------
P_ALVO = 40
df = MANIFESTOS[P_ALVO].copy()

# PADRONIZAÇÃO DA COLUNA 'FAIXA' (Limpa espaços e traduz longe/perto para far/near)
df['faixa'] = df['faixa'].astype(str).str.strip().str.lower()
df['faixa'] = df['faixa'].replace({'longe': 'far', 'perto': 'near'})

# Resgata o manifesto da HRV da memória (ou tenta carregar do disco caso não esteja)
if 'df_hrv_manifesto' not in globals():
    caminho_hrv = '/content/drive/MyDrive/TCC/csvs/manifesto_hrv_rqa.csv'
    if os.path.exists(caminho_hrv):
        df_hrv_manifesto = pd.read_csv(caminho_hrv)
    else:
        df_hrv_manifesto = None
        print("AVISO: manifesto_hrv_rqa.csv não encontrado na memória nem no disco.")

if df_hrv_manifesto is not None:
    df_hrv_manifesto = df_hrv_manifesto.copy()
    df_hrv_manifesto['faixa'] = df_hrv_manifesto['faixa'].astype(str).str.strip().str.lower()
    df_hrv_manifesto['faixa'] = df_hrv_manifesto['faixa'].replace({'longe': 'far', 'perto': 'near'})

print("="*60)
print("1. R-7: ANTECIPAÇÃO MEDIANA (Fuga da Banda Basal: μ ± 1.5σ)")
print("="*60)

tempos_alerta = {'DET': [], 'L_max': [], 'LAM': []}
pacientes = df['record'].unique()

# Fator de sensibilidade realista para captar alertas precoces
FATOR_SIGMA = 1.5

for p in pacientes:
    df_p = df[df['record'] == p]

    # Isola as fases pela coluna 'faixa' padronizada
    far = df_p[df_p['faixa'] == 'far']
    near = df_p[df_p['faixa'] == 'near'].sort_values('tempo_rel_vfon_min')

    if far.empty or near.empty:
        continue

    for feature in ['DET', 'L_max', 'LAM']:
        if feature not in df.columns:
            continue

        mu = far[feature].mean()
        sigma = far[feature].std()

        limite_sup = mu + (FATOR_SIGMA * sigma)
        limite_inf = mu - (FATOR_SIGMA * sigma)

        # Encontra pontos que furaram o teto OU o piso basal
        fugas = near[(near[feature] > limite_sup) | (near[feature] < limite_inf)]

        if not fugas.empty:
            # Pega o instante mais distante do colapso em segundos absolutos
            primeiro_cruzamento_s = abs(fugas.iloc[0]['tempo_rel_vfon_min'] * 60)
            tempos_alerta[feature].append(primeiro_cruzamento_s)

for f, tempos in tempos_alerta.items():
    if tempos:
        mediana = np.median(tempos)
        print(f"Métrica {f:>5s}: fuga definitiva a {mediana:.1f} segundos antes da FV (n={len(tempos)} pacientes).")
    else:
        print(f"Métrica {f:>5s}: nenhum paciente fugiu da banda basal de {FATOR_SIGMA}σ.")

print("\n"+"="*60)
print("2. BOOTSTRAP (IC 95%) PARA OS COHEN'S d PAREADOS (DADOS REAIS)")
print("="*60)

# Função auxiliar para calcular o d de Cohen Pareado (d_z)
def calc_dz(diferencas, axis=-1):
    std = np.std(diferencas, ddof=1, axis=axis)
    return np.divide(np.mean(diferencas, axis=axis), std, out=np.zeros_like(std), where=std!=0)

# ---------------------------------------------------------------------------
# Extração das diferenças reais (near - far) para a Amplitude (V_max)
# ---------------------------------------------------------------------------
dif_amp_real = []
for p in pacientes:
    df_p = df[df['record'] == p]
    far_mean = df_p[df_p['faixa'] == 'far']['V_max'].mean()
    near_mean = df_p[df_p['faixa'] == 'near']['V_max'].mean()

    if not np.isnan(far_mean) and not np.isnan(near_mean):
        dif_amp_real.append(near_mean - far_mean)

dif_amp_real = np.array(dif_amp_real)

if len(dif_amp_real) > 1:
    res_amp = bootstrap((dif_amp_real,), calc_dz, n_resamples=10000, confidence_level=0.95, method='BCa')
    print(f"Amplitude (V_max, n={len(dif_amp_real)}): dz = {calc_dz(dif_amp_real):.3f} --> IC 95% = [{res_amp.confidence_interval.low:.3f}, {res_amp.confidence_interval.high:.3f}]")
else:
    print("Dados insuficientes para calcular Bootstrap da Amplitude.")

# ---------------------------------------------------------------------------
# Extração das diferenças reais para HRV (LAM) usando o df da HRV
# ---------------------------------------------------------------------------
if df_hrv_manifesto is not None:
    dif_hrv_real = []
    pacientes_hrv = df_hrv_manifesto['record'].unique()

    for p in pacientes_hrv:
        df_p_hrv = df_hrv_manifesto[df_hrv_manifesto['record'] == p]
        far_mean_hrv = df_p_hrv[df_p_hrv['faixa'] == 'far']['LAM'].mean()
        near_mean_hrv = df_p_hrv[df_p_hrv['faixa'] == 'near']['LAM'].mean()

        if not np.isnan(far_mean_hrv) and not np.isnan(near_mean_hrv):
            dif_hrv_real.append(near_mean_hrv - far_mean_hrv)

    dif_hrv_real = np.array(dif_hrv_real)

    if len(dif_hrv_real) > 1:
        res_hrv = bootstrap((dif_hrv_real,), calc_dz, n_resamples=10000, confidence_level=0.95, method='BCa')
        print(f"HRV (LAM, n={len(dif_hrv_real)}): dz = {calc_dz(dif_hrv_real):.3f} --> IC 95% = [{res_hrv.confidence_interval.low:.3f}, {res_hrv.confidence_interval.high:.3f}]")
    else:
        print("Dados insuficientes para calcular Bootstrap da HRV.")
else:
    print("DataFrame da HRV não está disponível para calcular o IC.")

## Part 11 — Figure Regeneration (English labels)

In [ ]:
# =============================================================================
#  REGENERATION OF 4 TCC FIGURES — ALL LABELS IN ENGLISH, ALL TEXT IN BOLD
#  Paste each block below as a separate Colab cell, in order.
#  Requires: Part 0 already executed (DRIVE_TCC, CSV_DIR, IMG_DIR, FEATURES...)
# =============================================================================


# =============================================================================
#  CELL 0 · GLOBAL BOLD STYLE + HELPERS  (run this first, always)
# =============================================================================
import os, glob
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

# --- fallbacks in case Part 0 was not executed ---------------------------
DRIVE_TCC = globals().get('DRIVE_TCC', '/content/drive/MyDrive/TCC')
CSV_DIR   = globals().get('CSV_DIR',   os.path.join(DRIVE_TCC, 'csvs'))
IMG_DIR   = globals().get('IMG_DIR',   os.path.join(DRIVE_TCC, 'Imagens'))
os.makedirs(IMG_DIR, exist_ok=True)

FEATURES = ['RR','DET','L','L_max','DIV','L_entr','LAM','TT','V_max',
            'V_entr','W','W_max','W_entr','DET_RR','LAM_DET']

# --- EVERYTHING BOLD, EVERYWHERE ----------------------------------------
plt.rcParams.update({
    'figure.dpi'         : 110,
    'savefig.dpi'        : 300,
    'savefig.bbox'       : 'tight',
    'savefig.facecolor'  : 'white',
    'font.size'          : 13,
    'font.weight'        : 'bold',      # <- every text object is bold
    'axes.titlesize'     : 16,
    'axes.titleweight'   : 'bold',
    'axes.labelsize'     : 13,
    'axes.labelweight'   : 'bold',
    'xtick.labelsize'    : 11,
    'ytick.labelsize'    : 11,
    'figure.titlesize'   : 20,
    'figure.titleweight' : 'bold',
    'legend.fontsize'    : 12,
    'legend.frameon'     : True,
    'legend.framealpha'  : 0.95,
    'legend.edgecolor'   : '#222222',
    'lines.linewidth'    : 2.2,
    'axes.linewidth'     : 1.4,
    'axes.grid'          : True,
    'grid.alpha'         : 0.30,
    'grid.linestyle'     : '--',
    'axes.edgecolor'     : '#222222',
    'axes.labelcolor'    : '#111111',
    'mathtext.default'   : 'bf',
})

def bold_ticks(ax):
    """Force every tick label (the VALUES on both axes) to bold."""
    for lb in ax.get_xticklabels() + ax.get_yticklabels():
        lb.set_fontweight('bold')
    ot = ax.get_yaxis().get_offset_text();  ot.set_fontweight('bold')
    ot = ax.get_xaxis().get_offset_text();  ot.set_fontweight('bold')
    return ax

def bold_all(fig):
    """Safety net: sweeps the whole figure and bolds any remaining text."""
    for t in fig.findobj(mpl.text.Text):
        t.set_fontweight('bold')
    return fig

def csv_path(nome):
    """Looks for the CSV in TCC/csvs; falls back to the TCC root."""
    if not nome.endswith('.csv'):
        nome += '.csv'
    p1 = os.path.join(CSV_DIR, nome)
    p2 = os.path.join(DRIVE_TCC, nome)
    return p1 if os.path.exists(p1) else (p2 if os.path.exists(p2) else p1)

print("Bold style applied. CSV_DIR =", CSV_DIR, "| IMG_DIR =", IMG_DIR)




In [ ]:
# =============================================================================
#  CELL 1 · boxplots_15_features_p40.png
#           RQA metric distributions at fixed instants before collapse
# =============================================================================
P = 40                                   # percentile used for binarization

INSTANTS = [
    ('-60 min', -60*60),
    ('-30 min', -30*60),
    ('-10 min', -10*60),
    ('-2 min',   -2*60),
    ('-30 s',       -30),
    ('Collapse',      0),
]
TOL = 20                                  # +/- seconds around each instant

def windows_at(df_func, t_target, tol=TOL):
    if t_target == 0:
        return df_func[(df_func.tempo_rel_vfon_s >= 0) &
                       (df_func.tempo_rel_vfon_s <= 30)]
    return df_func[(df_func.tempo_rel_vfon_s >= t_target - tol) &
                   (df_func.tempo_rel_vfon_s <= t_target + tol)]

path_p40 = csv_path(f'manifesto_p{P:02d}.csv')
assert os.path.exists(path_p40), f"Not found: {path_p40}"
df = pd.read_csv(path_p40)
print(f"manifesto_p{P:02d}.csv loaded: {len(df)} windows")

fig, axes = plt.subplots(5, 3, figsize=(19, 24))
fig.suptitle(f'Distribution of the 15 RQA Metrics at Fixed Instants Before Collapse '
             f'(SDDB, p{P:02d})', fontsize=22, fontweight='bold', y=0.998)

labels     = [lbl for lbl, _ in INSTANTS]
p_median   = dict(color='black',   linewidth=3.0)
p_fliers   = dict(marker='.', color='gray', alpha=0.40, markersize=6)
p_lines    = dict(color='#222222', linewidth=1.5)

for ax, feat in zip(axes.flat, FEATURES):
    box_data = [windows_at(df, t)[feat].dropna().values for _, t in INSTANTS]

    bp = ax.boxplot(box_data, tick_labels=labels, showfliers=True,
                    patch_artist=True, widths=0.6,
                    medianprops=p_median, flierprops=p_fliers,
                    whiskerprops=p_lines, capprops=p_lines)

    colors = plt.cm.coolwarm(np.linspace(0, 1, len(INSTANTS)))
    for patch, c in zip(bp['boxes'], colors):
        patch.set_facecolor(c); patch.set_alpha(0.85)
        patch.set_edgecolor('#222222'); patch.set_linewidth(1.4)

    ax.set_title(feat, fontsize=17, fontweight='bold', pad=8)
    ax.set_xlabel('Time relative to collapse', fontsize=13, fontweight='bold')
    ax.set_ylabel(f'{feat} (a.u.)', fontsize=13, fontweight='bold')
    ax.tick_params(axis='x', rotation=45, labelsize=11)
    ax.tick_params(axis='y', labelsize=11)
    ax.grid(True, axis='y', linestyle='--', alpha=0.40)
    bold_ticks(ax)

    # per-panel 1-99% scaling (this is what fixes the flattened RR panel)
    allv = np.concatenate([d for d in box_data if len(d) > 0]) \
           if any(len(d) for d in box_data) else np.array([])
    if allv.size:
        y0, y1 = np.percentile(allv, [1, 99])
        if y1 > y0:
            m = (y1 - y0) * 0.15
            ax.set_ylim(y0 - m, y1 + m)
        else:
            ax.set_ylim(y0 - 1e-4, y1 + 1e-4)

for ax in axes.flat[len(FEATURES):]:
    ax.set_visible(False)

plt.tight_layout(rect=[0, 0, 1, 0.985])
bold_all(fig)
out = os.path.join(IMG_DIR, f'boxplots_15_features_p{P:02d}.png')
plt.savefig(out, dpi=300, bbox_inches='tight')
plt.show()
print("Saved:", out)




In [ ]:
# =============================================================================
#  CELL 2 · trajetorias_15_features_p40.png
#           Temporal trajectory (mean +/- SEM) of the 15 RQA metrics
#  NOTE: the original cell read manifesto_p10.csv but titled the figure "p40".
#        Fixed here — it now reads manifesto_p40.csv.
# =============================================================================
P = 40

path_p40 = csv_path(f'manifesto_p{P:02d}.csv')
assert os.path.exists(path_p40), f"Not found: {path_p40}"
df = pd.read_csv(path_p40)
df['bin'] = (df.tempo_rel_vfon_s // 60) * 60          # 1-minute bins

fig, axes = plt.subplots(5, 3, figsize=(19, 24))
fig.suptitle(f'Temporal Trajectory of the 15 RQA Metrics up to Collapse '
             f'(SDDB, p{P:02d})', fontsize=22, fontweight='bold', y=0.998)

for k, (ax, feat) in enumerate(zip(axes.flat, FEATURES)):
    traj = df.groupby('bin')[feat].agg(['mean', 'std', 'count']).reset_index()
    traj = traj[traj['count'] >= 3]
    sem  = traj['std'] / np.sqrt(traj['count'])

    ax.fill_between(traj.bin/60, traj['mean'] - sem, traj['mean'] + sem,
                    alpha=0.22, color='tab:blue',
                    label='Mean $\\pm$ SEM' if k == 0 else None)
    ax.plot(traj.bin/60, traj['mean'], color='darkblue', lw=2.4, zorder=3,
            label='Mean across patients' if k == 0 else None)
    ax.axvline(0,   color='red',    lw=2.5, alpha=0.55, zorder=1,
               label='Collapse (VF onset)' if k == 0 else None)
    ax.axvline(-30, color='orange', ls='--', lw=2.0,
               label='Far / near boundary (-30 min)' if k == 0 else None)

    ax.set_title(feat, fontsize=17, fontweight='bold', pad=8)
    ax.set_xlabel('Time relative to collapse (min)', fontsize=13, fontweight='bold')
    ax.set_ylabel(f'{feat} (a.u.)', fontsize=13, fontweight='bold')
    ax.tick_params(labelsize=11)
    ax.grid(True, alpha=0.35, linestyle='--')
    bold_ticks(ax)

for ax in axes.flat[len(FEATURES):]:
    ax.set_visible(False)

handles, labs = axes.flat[0].get_legend_handles_labels()
if handles:
    leg = fig.legend(handles, labs, loc='upper right', ncol=2, fontsize=14,
                     bbox_to_anchor=(0.995, 0.988))
    for t in leg.get_texts():
        t.set_fontweight('bold')

plt.tight_layout(rect=[0, 0, 1, 0.978])
bold_all(fig)
out = os.path.join(IMG_DIR, f'trajetorias_15_features_p{P:02d}.png')
plt.savefig(out, dpi=300, bbox_inches='tight')
plt.show()
print("Saved:", out)




In [ ]:
# =============================================================================
#  CELL 3 · contraste_dominio_amplitude_vs_hrv.png
#           Cohen's d per patient: ECG amplitude domain vs HRV domain
# =============================================================================
P_AMP = 10        # amplitude manifest used in the original analysis (10 or 40)

def d_per_patient(df_in, features, col_band='faixa'):
    """Cohen's d (far vs near), ONE value per patient per band."""
    rows = []
    for feat in features:
        if feat not in df_in.columns:
            continue
        sub = df_in[df_in[col_band].isin(['longe', 'perto'])]
        agg = sub.groupby(['record', col_band])[feat].mean().reset_index()
        counts = agg.groupby('record')[col_band].nunique()
        complete = counts[counts == 2].index
        if len(complete) < 3:
            continue
        ok = agg[agg['record'].isin(complete)].sort_values('record')
        a = ok[ok[col_band] == 'longe'][feat].values     # FAR from collapse
        b = ok[ok[col_band] == 'perto'][feat].values     # NEAR collapse
        sp = np.sqrt(((len(a)-1)*a.std(ddof=1)**2 + (len(b)-1)*b.std(ddof=1)**2)
                     / (len(a)+len(b)-2))
        rows.append({'feature': feat, 'n_patients': len(complete),
                     'cohen_d': round(abs((a.mean()-b.mean())/sp), 3) if sp > 0 else np.nan,
                     'mean_far': round(a.mean(), 4),
                     'mean_near': round(b.mean(), 4)})
    return pd.DataFrame(rows).sort_values('cohen_d', ascending=False)

# ---- AMPLITUDE arm ----
df_amp = pd.read_csv(csv_path(f'manifesto_p{P_AMP:02d}.csv'))
d_amp  = d_per_patient(df_amp, FEATURES)

# ---- HRV arm ----
df_hrv = pd.read_csv(csv_path('manifesto_hrv_rqa.csv'))
if 'aprovada' in df_hrv.columns:
    df_hrv = df_hrv[df_hrv.aprovada]
d_hrv = d_per_patient(df_hrv, FEATURES)

comp = (d_amp[['feature', 'cohen_d']].rename(columns={'cohen_d': 'd_AMPLITUDE'})
        .merge(d_hrv[['feature', 'cohen_d']].rename(columns={'cohen_d': 'd_HRV'}),
               on='feature', how='outer')
        .sort_values('d_HRV', ascending=False))
print(comp.to_string(index=False))

fig, ax = plt.subplots(figsize=(15, 8))
y, h = np.arange(len(comp)), 0.38

b1 = ax.barh(y + h/2, comp.d_AMPLITUDE, h, label='ECG amplitude (m = 1)',
             color='#9aa5b1', edgecolor='#222222', linewidth=1.4)
b2 = ax.barh(y - h/2, comp.d_HRV, h, label='HRV / tachogram (Takens, m > 1)',
             color='#1f4e79', edgecolor='#222222', linewidth=1.4)

ax.axvline(0.2, color='darkorange', ls='--', lw=2.5, label='d = 0.2 (small effect)')
ax.axvline(0.5, color='crimson',    ls='--', lw=2.5, label='d = 0.5 (medium effect)')

# numeric value at the end of every bar — bold
for bars in (b1, b2):
    for bar in bars:
        w = bar.get_width()
        if np.isfinite(w):
            ax.text(w + 0.012, bar.get_y() + bar.get_height()/2, f'{w:.2f}',
                    va='center', ha='left', fontsize=11, fontweight='bold',
                    color='#111111')

ax.set_yticks(y)
ax.set_yticklabels(comp.feature, fontsize=13, fontweight='bold')
ax.set_xlabel("Cohen's d per patient — far vs near collapse",
              fontsize=15, fontweight='bold')
ax.set_ylabel('RQA metric', fontsize=15, fontweight='bold')
ax.set_title("Cohen's d per Patient by RQA Metric: ECG Amplitude vs HRV Domain",
             fontsize=19, fontweight='bold', pad=14)
leg = ax.legend(fontsize=13, loc='lower right')
for t in leg.get_texts():
    t.set_fontweight('bold')
ax.grid(True, axis='x', alpha=0.35, ls='--')
ax.set_xlim(0, max(0.65, np.nanmax([comp.d_AMPLITUDE.max(), comp.d_HRV.max()]) * 1.18))
bold_ticks(ax)

plt.tight_layout()
bold_all(fig)
out = os.path.join(IMG_DIR, 'contraste_dominio_amplitude_vs_hrv.png')
plt.savefig(out, dpi=300, bbox_inches='tight')
plt.show()
print("Saved:", out)
print(f"Max d AMPLITUDE = {comp.d_AMPLITUDE.max():.3f} | "
      f"Max d HRV = {comp.d_HRV.max():.3f}  (n = 7 patients — descriptive, not inferential)")




In [ ]:
# =============================================================================
#  CELL 4 · rp_tres_estagios.png
#           Recurrence plots at three stages: normal / pre-VF / VF
#           Needs the raw record '30' (.hea + .dat) reachable by wfdb.rdsamp
# =============================================================================
import wfdb
from scipy.spatial.distance import pdist, squareform
from scipy.signal import butter, filtfilt

RECORD_ID = '30'
FS_LOC    = 250
VFON_S    = 28473                      # VF onset at 07:54:33
i_vfon    = int(VFON_S * FS_LOC)
N5S       = 5 * FS_LOC                 # 1250 samples = 5 s

def limpar_nan(x):
    x = np.asarray(x, dtype=float)
    bad = ~np.isfinite(x)
    if bad.any():
        idx, ok = np.arange(len(x)), ~bad
        if ok.sum() < 2:
            return np.zeros_like(x)
        x = x.copy(); x[bad] = np.interp(idx[bad], idx[ok], x[ok])
    return x

def filtrar_ecg(sinal, fs=FS_LOC, lowcut=0.5, highcut=40.0, ordem=3):
    """Zero-phase Butterworth band-pass, NaN-robust (same as Section 3.1)."""
    sinal = limpar_nan(sinal)
    nyq = 0.5 * fs
    b, a = butter(ordem, [lowcut/nyq, highcut/nyq], btype='band')
    return filtfilt(b, a, sinal)

# --- locate the record wherever it lives (cwd, matrizes/30, SDDB) ---------
def find_record(rec):
    for cand in ([rec, os.path.join(DRIVE_TCC, 'matrizes', rec, rec),
                  os.path.join(DRIVE_TCC, 'SDDB', rec)]
                 + [p[:-4] for p in glob.glob(os.path.join(DRIVE_TCC, '**', f'{rec}.hea'),
                                              recursive=True)]):
        if os.path.exists(cand + '.hea'):
            return cand
    raise FileNotFoundError(f"{rec}.hea not found under {DRIVE_TCC} or cwd")

REC = find_record(RECORD_ID)
print("Reading record from:", REC)

i_norm = i_vfon - 10*60*FS_LOC                  # 10 min before collapse
i_pre  = i_vfon - N5S                           # last 5 s before collapse
i_fv   = i_vfon + 2*FS_LOC                      # 2 s after onset (sustained VF)

sig_n, _ = wfdb.rdsamp(REC, sampfrom=i_norm, sampto=i_norm + N5S)
sig_p, _ = wfdb.rdsamp(REC, sampfrom=i_pre,  sampto=i_pre  + N5S)
sig_f, _ = wfdb.rdsamp(REC, sampfrom=i_fv,   sampto=i_fv   + N5S)

ecg_n = filtrar_ecg(sig_n[:, 0])
ecg_p = filtrar_ecg(sig_p[:, 0])
ecg_f = filtrar_ecg(sig_f[:, 0])

# epsilon calibrated ONCE on the healthy rhythm and reused in all three stages
EPS = np.std(ecg_n) * 0.5
print(f"Fixed threshold (epsilon) = {EPS:.4f}")

def recurrence(x, eps):
    return (squareform(pdist(x.reshape(-1, 1), metric='euclidean')) < eps).astype(int)

rp_n, rp_p, rp_f = recurrence(ecg_n, EPS), recurrence(ecg_p, EPS), recurrence(ecg_f, EPS)

t_ms = np.arange(N5S) / FS_LOC                   # seconds within the window

fig, axes = plt.subplots(3, 2, figsize=(17, 14))
fig.suptitle(f'From Normal Rhythm to Ventricular Fibrillation: '
             f'ECG and Recurrence Plots (Patient {RECORD_ID})',
             fontsize=21, fontweight='bold', y=0.985)

stages = [
    (ecg_n, rp_n, 'green',  '1. Normal ECG (10 min before collapse)',
     'Recurrence Plot — Normal (Diagonal Lines / Order)'),
    (ecg_p, rp_p, 'orange', '2. Pre-VF ECG (last 5 s before collapse)',
     'Recurrence Plot — Pre-VF (Blocks / Instability)'),
    (ecg_f, rp_f, 'red',    '3. VF ECG (after collapse onset)',
     'Recurrence Plot — VF (Sustained Fibrillation Structure)'),
]

for row, (ecg, rp, color, t_left, t_right) in enumerate(stages):
    ax = axes[row, 0]
    ax.plot(t_ms, ecg, color=color, lw=1.6)
    ax.set_title(t_left, fontsize=16, fontweight='bold')
    ax.set_ylabel('Amplitude (mV)', fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.35, ls='--')
    bold_ticks(ax)

    ax = axes[row, 1]
    ax.imshow(rp, cmap='binary', origin='lower',
              extent=[0, 5, 0, 5], aspect='equal')
    ax.set_title(t_right, fontsize=16, fontweight='bold')
    ax.set_ylabel('Time (s)', fontsize=14, fontweight='bold')
    ax.grid(False)
    bold_ticks(ax)

axes[2, 0].set_xlabel('Time (s)', fontsize=14, fontweight='bold')
axes[2, 1].set_xlabel('Time (s)', fontsize=14, fontweight='bold')

plt.tight_layout(rect=[0, 0.01, 1, 0.962])
bold_all(fig)
out = os.path.join(IMG_DIR, 'rp_tres_estagios.png')
plt.savefig(out, dpi=300, bbox_inches='tight')
plt.show()
print("Saved:", out)